# Loans at Risk: Capturing Default — Modeling

## Purpose

This notebook implements the modeling stage of the *Loans at Risk: Capturing Default* project.

The analysis uses the `feature_base` datasets produced in the ETL pipeline:

- **Training set:** loans issued between 2007 and 2014  
- **Testing set:** loans issued in 2015  

The modeling population is restricted to loans with terminal outcomes in order to define a clear prediction target.

The purpose of this notebook is to train and evaluate a small set of representative predictive models on this population and identify the model that produces the most accurate predictions of borrower default risk.

---

## Analytical Approach

Default prediction is formulated as a binary classification problem.

Loans are classified as:

- **Default (1):** Charged Off or Default  
- **Non-default (0):** Fully Paid  

Two policy-related loan status categories are normalized before modeling:

- “Does not meet the credit policy. Status: Fully Paid” → Fully Paid  
- “Does not meet the credit policy. Status: Charged Off” → Charged Off  

This preserves the economic outcome of the loan while removing administrative distinctions that are not relevant to the prediction task.

Model performance is evaluated primarily using **ROC-AUC**, with additional metrics including precision, recall, F1 score, and confusion matrices. Performance is compared between the training and testing datasets to assess predictive accuracy and potential overfitting.

---

## Model Strategy

Three models are evaluated in this analysis:

1. **Logistic Regression**  
2. **Random Forest**  
3. **CatBoost**

These models represent three different approaches to predictive modeling: a classical statistical model, a bagging-based tree ensemble, and a boosting-based ensemble method.

### Logistic Regression

Logistic Regression serves as the baseline model and represents the traditional statistical approach widely used in credit risk modeling (Thomas et al., 2017).

The model combines borrower characteristics into a weighted linear score. Each feature contributes positively or negatively depending on how strongly it is associated with default risk. Because this score can take any value, it is transformed using the **logistic function**, which maps the score onto a value between 0 and 1. The result can therefore be interpreted as the predicted probability that a loan will default. Logistic regression has historically been the dominant methodology used in credit scoring because it produces stable probability estimates and remains relatively interpretable compared to more complex machine learning models. It therefore provides a natural benchmark against which more flexible nonlinear models can be evaluated.

### Random Forest

Random Forest represents the **bagging ensemble paradigm** introduced by Breiman (2001).

A decision tree predicts outcomes by repeatedly dividing the data into smaller groups based on feature values. Each split attempts to separate loans with different outcomes—for example, separating borrowers with high debt ratios from those with lower ones. A single decision tree can be unstable because small changes in the data may lead to very different splits. Random Forest addresses this by building **many trees**, each trained on a slightly different random sample of the data and predictor variables. Each tree produces its own prediction, and the final model output is obtained by averaging the predictions across all trees. Combining many trees in this way reduces the instability of individual trees while allowing the model to capture nonlinear relationships and interactions between variables.

### CatBoost

CatBoost represents the **boosting ensemble paradigm**, where models are trained sequentially rather than independently (Prokhorenkova et al., 2018).

Like Random Forest, CatBoost uses decision trees as its building blocks. However, instead of training many independent trees, boosting trains trees **one after another**. Each new tree focuses on correcting prediction errors made by the previous trees. Over many iterations the model gradually improves its predictions by concentrating on the observations that were most difficult to predict earlier in the training process. CatBoost also includes techniques designed for tabular datasets containing categorical variables and aims to reduce bias during model training. Empirical research shows that tree-based ensembles remain effective approaches for tabular prediction tasks (Grinsztajn et al., 2022).

---

Together these models represent three distinct learning approaches:

- linear statistical modeling  
- bagged tree ensembles  
- boosted tree ensembles  

Their performance is compared in order to identify the model that produces the most accurate predictions of borrower default risk.

---

## Structure

The notebook proceeds in four stages.

1. **Modeling Population**  
   The modeling population is finalized and the binary target variable is constructed.

2. **Feature Engineering**  
   Additional feature transformations are performed to prepare the dataset for model training.

3. **Model Training**  
   Logistic Regression, Random Forest, and CatBoost models are trained on the training dataset.

4. **Model Evaluation**  
   Model performance is evaluated on both the training and testing datasets in order to identify the best-performing model.

---

In [1]:
from pathlib import Path
import sys

current_path = Path.cwd().resolve()
project_root = None

for parent_path in (current_path, *current_path.parents):
    if (parent_path / "pyproject.toml").exists():
        project_root = parent_path
        break

if project_root is None:
    raise RuntimeError(
        f"Failed to resolve project root: pyproject.toml not found from {current_path}"
    )

src_path = project_root / "src"
if not src_path.exists():
    raise RuntimeError(f"Expected 'src' directory at: {src_path}")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

{
    "stage": "bootstrap_import_path_ready",
    "project_root": str(project_root),
}

{'stage': 'bootstrap_import_path_ready',
 'project_root': 'D:\\Portfolio\\loans-at-risk-capturing-default'}

In [2]:
# ===========================================
# Imports: libraries and project modules
# ===========================================

from datetime import datetime, timezone
from typing import Callable
import uuid
import json

from IPython.display import display
import pandas as pd

import config.logging as log_config
import analysis.dataset_artifacts as da
import analysis.modeling_artifacts as ma
import plots.report_figures as rf
import features.modeling_population as mp
import features.feature_utils as fu
import features.feature_engineering as fe
import modeling.train_models as tm
import modeling.evaluate_models as em
import modeling.save_model as sm


In [3]:
# ===============================
# Paths and run context
# ===============================

logs_root = project_root / "logs"
logs_root.mkdir(parents=True, exist_ok=True)

PROJECT_LOG_FILE = logs_root / "project.log"

run_id = uuid.uuid4().hex[:10]
run_timestamp_utc = datetime.now(timezone.utc)

run_header = (
    "NEW RUN: "
    f"{run_timestamp_utc.strftime('%Y-%m-%d %H:%M:%S')} UTC | "
    f"RUN_ID={run_id}"
)

log_config.log_messages("\n" + "=" * 60, PROJECT_LOG_FILE)
log_config.log_messages(run_header, PROJECT_LOG_FILE)
log_config.log_messages("=" * 60 + "\n", PROJECT_LOG_FILE)

log: Callable[[str], None] = log_config.get_logger(PROJECT_LOG_FILE)
log("Modeling notebook initialized")
log(f"project_root: {project_root}")
log(run_header)

{
    "stage": "run_started",
    "run_id": run_id,
    "utc_timestamp": run_timestamp_utc.isoformat(),
}

# ---------------------------------------------------------------
# Inputs for this notebook (processed, report)
# ---------------------------------------------------------------

feature_base_train_data_file = project_root / "data" / "interim" / "feature_base_loan_data_2007_2014.parquet"
feature_base_test_data_file = project_root / "data" / "interim" / "feature_base_loan_data_2015.parquet"

shared_model_train_data_file = project_root / "data" / "processed" / "shared_model_train_data.parquet"
shared_model_test_data_file = project_root / "data" / "processed" / "shared_model_test_data.parquet"

tree_no_log_train_data_file = project_root / "data" / "processed" / "tree_no_log_train_data.parquet"
tree_no_log_test_data_file = project_root / "data" / "processed" / "tree_no_log_test_data.parquet"

catboost_native_train_data_file = project_root / "data" / "processed" / "catboost_native_train_data.parquet"
catboost_native_test_data_file = project_root / "data" / "processed" / "catboost_native_test_data.parquet"

selected_model_input_train_file = project_root / "data" / "processed" / "selected_model_input_train_data.parquet"
selected_model_input_test_file = project_root / "data" / "processed" / "selected_model_input_test_data.parquet"

processed_data_dir = project_root / "data" / "processed"
metadata_file = processed_data_dir / "selected_model_metadata.json"

artifacts_dir = project_root / "artifacts"
modeling_artifacts_dir = artifacts_dir / "modeling"
modeling_tables_dir = modeling_artifacts_dir / "tables"
modeling_figures_dir = modeling_artifacts_dir / "figures"

processed_data_dir.mkdir(parents=True, exist_ok=True)
modeling_tables_dir.mkdir(parents=True, exist_ok=True)
modeling_figures_dir.mkdir(parents=True, exist_ok=True)

model_dir = project_root / "models"
selected_model_output_path = model_dir / "catboost_shared_tuned.joblib"
model_dir.mkdir(parents=True, exist_ok=True)


log(f"Shared model train parquet path: {shared_model_train_data_file}")
log(f"Shared model test parquet path: {shared_model_test_data_file}")
log(f"Tree no-log train parquet path: {tree_no_log_train_data_file}")
log(f"Tree no-log test parquet path: {tree_no_log_test_data_file}")
log(f"CatBoost native train parquet path: {catboost_native_train_data_file}")
log(f"CatBoost native test parquet path: {catboost_native_test_data_file}")
log(f"Modeling tables directory: {modeling_tables_dir}")
log(f"Modeling figures directory: {modeling_figures_dir}")


## 1. Modeling Population

In [4]:
# --------------------------------------------------------
# Load feature_base datasets + checkpoint
# --------------------------------------------------------

df_feature_base_train = pd.read_parquet(feature_base_train_data_file)
df_feature_base_test = pd.read_parquet(feature_base_test_data_file)

feature_base_overview_df = da.build_dataset_overview(
    df_train=df_feature_base_train,
    df_test=df_feature_base_test,
)

display(feature_base_overview_df)
log(f"[feature_base] overview: {feature_base_overview_df.to_dict(orient='records')}")

,dataset_split,rows,columns,memory_mb
0,train,466287,66,206.33
1,test,421095,66,186.34


In [5]:
# --------------------------------------------------------
# Inspect loan_status distribution in feature_base
# --------------------------------------------------------

loan_status_overview_df = pd.DataFrame(
    {
        "train": df_feature_base_train["loan_status"].value_counts(dropna=False),
        "test": df_feature_base_test["loan_status"].value_counts(dropna=False),
    }
).fillna(0).astype(int)

display(loan_status_overview_df)

log(f"[feature_base] loan_status distribution inspected")

,train,test
loan_status,,
charged_off,52474,11075
current,173578,346702
default,111,93
does_not_meet_the_credit_policy._status:charged_off,761,0
does_not_meet_the_credit_policy._status:fully_paid,1988,0
fully_paid,227611,48844
in_grace_period,3526,5122
late_(16-30_days),1109,1711
late_(31-120_days),5129,7548


In [6]:
# --------------------------------------------------------
# Build terminal cohort
# --------------------------------------------------------

terminal_statuses = [
    "fully_paid",
    "charged_off",
    "default",
    "does_not_meet_the_credit_policy._status:fully_paid",
    "does_not_meet_the_credit_policy._status:charged_off",
]

df_model_population_train = mp.build_terminal_cohort(
    df=df_feature_base_train,
    status_column="loan_status",
    terminal_statuses=terminal_statuses,
    log=PROJECT_LOG_FILE,
)

df_model_population_test = mp.build_terminal_cohort(
    df=df_feature_base_test,
    status_column="loan_status",
    terminal_statuses=terminal_statuses,
    log=PROJECT_LOG_FILE,
)

{
    "stage": "terminal_cohort_built",
    "train_rows": int(df_model_population_train.shape[0]),
    "test_rows": int(df_model_population_test.shape[0]),
    "train_columns": int(df_model_population_train.shape[1]),
    "test_columns": int(df_model_population_test.shape[1]),
}

{'stage': 'terminal_cohort_built',
 'train_rows': 282945,
 'test_rows': 60012,
 'train_columns': 66,
 'test_columns': 66}

In [7]:
# --------------------------------------------------------
# Modeling population summary
# --------------------------------------------------------

modeling_population_summary_df = ma.build_modeling_population_summary_table(
    df_feature_base_train=df_feature_base_train,
    df_feature_base_test=df_feature_base_test,
    df_model_train=df_model_population_train,
    df_model_test=df_model_population_test,
    train_label="train",
    test_label="test",
    log=PROJECT_LOG_FILE,
)

modeling_population_summary_output_path = modeling_tables_dir / "modeling_population_summary.csv"
modeling_population_summary_df.to_csv(
    modeling_population_summary_output_path,
    index=False,
)


log(f"[model_population] summary written to: {modeling_population_summary_output_path}")


{
    "stage": "modeling_population_summary_created",
    "rows": int(modeling_population_summary_df.shape[0]),
    "output_path": str(modeling_population_summary_output_path),
}

{'stage': 'modeling_population_summary_created',
 'rows': 4,
 'output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\modeling_population_summary.csv'}

In [8]:
# --------------------------------------------------------
# Create binary target
# --------------------------------------------------------

positive_statuses = [
    "charged_off",
    "default",
    "does_not_meet_the_credit_policy._status:charged_off",
]

df_model_population_train = mp.create_target_default(
    df=df_model_population_train,
    status_column="loan_status",
    positive_statuses=positive_statuses,
    target_column="target_default",
    log=PROJECT_LOG_FILE,
)

df_model_population_test = mp.create_target_default(
    df=df_model_population_test,
    status_column="loan_status",
    positive_statuses=positive_statuses,
    target_column="target_default",
    log=PROJECT_LOG_FILE,
)

{
    "stage": "target_created",
    "target_column": "target_default",
    "train_target_non_null": int(df_model_population_train["target_default"].notna().sum()),
    "test_target_non_null": int(df_model_population_test["target_default"].notna().sum()),
}

{'stage': 'target_created',
 'target_column': 'target_default',
 'train_target_non_null': 282945,
 'test_target_non_null': 60012}

In [9]:
# --------------------------------------------------------
# Target distribution artifact
# --------------------------------------------------------

target_distribution_df = ma.build_target_distribution_table(
    df_train=df_model_population_train,
    df_test=df_model_population_test,
    target_column="target_default",
    train_label="train",
    test_label="test",
    log=PROJECT_LOG_FILE,
)

target_distribution_output_path = modeling_tables_dir / "target_distribution.csv"

target_distribution_df.to_csv(
    target_distribution_output_path,
    index=False,
)

log(f"[target_distribution] artifact written to: {target_distribution_output_path}")

{
    "stage": "target_distribution_created",
    "rows": int(target_distribution_df.shape[0]),
    "output_path": str(target_distribution_output_path),
}

{'stage': 'target_distribution_created',
 'rows': 2,
 'output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\target_distribution.csv'}

In [10]:
# --------------------------------------------------------
# Create dataframes for feature engineering
# --------------------------------------------------------

df_feature_engineering_train = df_model_population_train.copy()
df_feature_engineering_test = df_model_population_test.copy()

display(f"'target_default' in train: {'target_default' in df_feature_engineering_train.columns}")
display(f"'target_default' in test: {'target_default' in df_feature_engineering_test.columns}")

"'target_default' in train: True"

"'target_default' in test: True"


## Modeling Population – Conclusion

This stage restricts the dataset to loans with **terminal repayment outcomes** and constructs the binary prediction target `target_default`.

Loans in active repayment states (e.g., current, late, or in grace period) are excluded because their final credit outcome is not yet observed. Policy-related loan statuses indicating that a loan did not meet LendingClub’s internal credit policy are interpreted according to their underlying economic outcomes when constructing the target variable.

The resulting datasets therefore contain only loans with **realized repayment outcomes** and a clearly defined binary target suitable for predictive modeling.

Population reduction and target balance are documented in the modeling artifacts produced in this stage.

To establish a clear pipeline boundary, the resulting datasets are persisted as:

- `model_train_data.parquet`
- `model_test_data.parquet`

These files represent the finalized **modeling population** and serve as the input for the next stage of the pipeline.

---

The analysis now proceeds to **feature engineering**, where the submission-time feature space is prepared for model training.

## 2. Feature Engineering

In [11]:
# --------------------------------------------------------
# Validate modeling datasets
# --------------------------------------------------------

target_column = "target_default"

train_columns = set(df_feature_engineering_train.columns)
test_columns = set(df_feature_engineering_test.columns)

train_only_columns = sorted(train_columns - test_columns)
test_only_columns = sorted(test_columns - train_columns)

if target_column not in df_feature_engineering_train.columns:
    raise KeyError(f"'{target_column}' not found in train data.")

if target_column not in df_feature_engineering_test.columns:
    raise KeyError(f"'{target_column}' not found in test data.")

validation_checkpoint = {
    "target_column": target_column,
    "train_shape": df_feature_engineering_train.shape,
    "test_shape": df_feature_engineering_test.shape,
    "train_column_count": len(df_feature_engineering_train.columns),
    "test_column_count": len(df_feature_engineering_test.columns),
    "train_only_columns": train_only_columns,
    "test_only_columns": test_only_columns,
}

log(f"[model_dataset_validation] {validation_checkpoint}")

validation_checkpoint

{'target_column': 'target_default',
 'train_shape': (282945, 67),
 'test_shape': (60012, 67),
 'train_column_count': 67,
 'test_column_count': 67,
 'train_only_columns': [],
 'test_only_columns': []}

In [12]:
# --------------------------------------------------------
# Build feature group audit table for modeling datasets
# --------------------------------------------------------

feature_group_audit_df = fe.build_feature_group_audit(
    df_train=df_feature_engineering_train,
    df_test=df_feature_engineering_test,
    target_column=target_column,
    log=PROJECT_LOG_FILE,
)

display(
    feature_group_audit_df.loc[
        feature_group_audit_df["feature_group"] == "categorical",
        ["feature_name", "train_non_null_unique_values", "test_non_null_unique_values"]
    ].sort_values(by="train_non_null_unique_values", ascending=False)
)

display(
    feature_group_audit_df.loc[
        (feature_group_audit_df["feature_group"] == "categorical") &
        (feature_group_audit_df["train_non_null_unique_values"] > 20)
    ]
)

display(
    feature_group_audit_df.loc[
        feature_group_audit_df["is_binary"],
        ["feature_name", "feature_group"]
    ]
)

display(
    feature_group_audit_df.loc[
        feature_group_audit_df["requires_manual_review"]
    ]
)

,feature_name,train_non_null_unique_values,test_non_null_unique_values
2,purpose,14,14
1,loan_status,5,3
0,home_ownership,4,3


,feature_name,train_dtype,test_dtype,feature_group,is_binary,train_non_null_unique_values,test_non_null_unique_values,requires_manual_review


,feature_name,feature_group
15,has_mths_since_last_delinq,numeric
16,has_mths_since_last_major_derog,numeric
17,has_mths_since_last_record,numeric
18,has_mths_since_recent_bc_dlq,numeric
19,has_mths_since_recent_revol_delinq,numeric
57,term_months,numeric


,feature_name,train_dtype,test_dtype,feature_group,is_binary,train_non_null_unique_values,test_non_null_unique_values,requires_manual_review


In [13]:
# --------------------------------------------------------
# Save feature group audit artifact
# --------------------------------------------------------

feature_group_audit_file = modeling_tables_dir / "feature_group_audit.csv"

feature_group_audit_df.to_csv(feature_group_audit_file, index=False)

log(f"[feature_engineering][feature_group_audit] table_file={feature_group_audit_file}")

{
    "feature_group_audit_rows": int(feature_group_audit_df.shape[0]),
    "feature_group_audit_columns": feature_group_audit_df.columns.tolist(),
    "feature_group_audit_file": str(feature_group_audit_file),
}

{'feature_group_audit_rows': 66,
 'feature_group_audit_columns': ['feature_name',
  'train_dtype',
  'test_dtype',
  'feature_group',
  'is_binary',
  'train_non_null_unique_values',
  'test_non_null_unique_values',
  'requires_manual_review'],
 'feature_group_audit_file': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\feature_group_audit.csv'}

In [14]:
# --------------------------------------------------------
# Define feature groups for transformation
# --------------------------------------------------------

columns_to_drop = [
    "loan_status",
    "row_id",
]

categorical_features = [
    "purpose",
    "home_ownership",
]

binary_features = [
    "has_mths_since_last_delinq",
    "has_mths_since_last_major_derog",
    "has_mths_since_last_record",
    "has_mths_since_recent_bc_dlq",
    "has_mths_since_recent_revol_delinq",
]

numeric_features = [
    feature_name
    for feature_name in feature_group_audit_df["feature_name"].tolist()
    if feature_name not in columns_to_drop
    and feature_name not in categorical_features
    and feature_name not in binary_features
]

feature_group_checkpoint = {
    "columns_to_drop_count": len(columns_to_drop),
    "categorical_feature_count": len(categorical_features),
    "binary_feature_count": len(binary_features),
    "numeric_feature_count": len(numeric_features),
}

log(f"[feature_engineering][feature_groups] {feature_group_checkpoint}")

feature_group_checkpoint

{'columns_to_drop_count': 2,
 'categorical_feature_count': 2,
 'binary_feature_count': 5,
 'numeric_feature_count': 57}

In [15]:
# --------------------------------------------------------
# Validate feature group coverage
# --------------------------------------------------------

all_accounted_columns = (
    set(columns_to_drop)
    | set(categorical_features)
    | set(binary_features)
    | set(numeric_features)
    | {target_column}
)

if all_accounted_columns != set(df_feature_engineering_train.columns):
    missing_columns = sorted(set(df_feature_engineering_train.columns) - all_accounted_columns)
    extra_columns = sorted(all_accounted_columns - set(df_feature_engineering_train.columns))
    raise ValueError(
        f"Feature group coverage mismatch. Missing={missing_columns}, Extra={extra_columns}"
    )

feature_group_validation_checkpoint = {
    "feature_group_validation_passed": True,
    "accounted_feature_columns": len(all_accounted_columns),
    "train_total_columns": len(df_feature_engineering_train.columns),
    "missing_columns": [],
    "extra_columns": [],
    "target_column": target_column,
}

log(f"[feature_engineering][feature_group_validation] {feature_group_validation_checkpoint}")

feature_group_validation_checkpoint

{'feature_group_validation_passed': True,
 'accounted_feature_columns': 67,
 'train_total_columns': 67,
 'missing_columns': [],
 'extra_columns': [],
 'target_column': 'target_default'}

In [16]:
#--------------------------------------------------------
# Build numeric skewness summary table
#--------------------------------------------------------

numeric_skewness_summary_df = ma.build_numeric_skewness_summary(
    df_train=df_feature_engineering_train,
    numeric_features=numeric_features,
    log=PROJECT_LOG_FILE,
)

numeric_skewness_summary_file = modeling_tables_dir / "numeric_skewness_summary.csv"

numeric_skewness_summary_df.to_csv(
    numeric_skewness_summary_file,
    index=False,
)

log(f"[feature_engineering][numeric_skewness_summary] table_file={numeric_skewness_summary_file}")

{
    "numeric_skewness_summary_rows": int(numeric_skewness_summary_df.shape[0]),
    "numeric_skewness_summary_columns": numeric_skewness_summary_df.columns.tolist(),
    "numeric_skewness_summary_file": str(numeric_skewness_summary_file),
}

{'numeric_skewness_summary_rows': 57,
 'numeric_skewness_summary_columns': ['feature_name',
  'non_null_count',
  'skewness',
  'min_value',
  'median_value',
  'mean_value',
  'max_value'],
 'numeric_skewness_summary_file': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\numeric_skewness_summary.csv'}

In [17]:
# -------------------------------------------------------------------------------------------
# Show examples of features with high skewness that may benefit from log transformation
# -------------------------------------------------------------------------------------------

report_log_transform_feature_examples = [
    "annual_inc",
    "loan_amnt",
    "revol_bal",
    "tot_cur_bal",
]

log_transformation_comparison_figure_file = (
    modeling_figures_dir / "log_transformation_comparison.png"
)

rf.plot_log_transformation_comparison_figure(
    df_train=df_feature_engineering_train,
    feature_names=report_log_transform_feature_examples,
    output_path=log_transformation_comparison_figure_file,
    log=PROJECT_LOG_FILE,
)

log(
    "[feature_engineering][log_transformation_comparison_figure] "
    f"figure_file={log_transformation_comparison_figure_file}"
)

{
    "log_transformation_comparison_feature_count": len(report_log_transform_feature_examples),
    "log_transformation_comparison_features": report_log_transform_feature_examples,
    "log_transformation_comparison_figure_file": str(log_transformation_comparison_figure_file),
}

{'log_transformation_comparison_feature_count': 4,
 'log_transformation_comparison_features': ['annual_inc',
  'loan_amnt',
  'revol_bal',
  'tot_cur_bal'],
 'log_transformation_comparison_figure_file': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\figures\\log_transformation_comparison.png'}

In [18]:
#------------------------------------------------------------------------------------
# Define feature groups for transformation based on audit and skewness analysis
#------------------------------------------------------------------------------------

columns_to_drop = [
    "loan_status",
]

categorical_features = [
    "purpose",
    "home_ownership",
]

binary_features = [
    "has_mths_since_last_delinq",
    "has_mths_since_last_major_derog",
    "has_mths_since_last_record",
    "has_mths_since_recent_bc_dlq",
    "has_mths_since_recent_revol_delinq",
]

numeric_features = [
    feature_name
    for feature_name in feature_group_audit_df["feature_name"].tolist()
    if feature_name not in columns_to_drop
    and feature_name not in categorical_features
    and feature_name not in binary_features
]

log_transform_features = [
    "annual_inc",
    "loan_amnt",
    "revol_bal",
    "tot_coll_amt",
    "total_bal_ex_mort",
    "avg_cur_bal",
    "bc_open_to_buy",
    "tot_cur_bal",
    "tot_hi_cred_lim",
    "total_bc_limit",
    "total_il_high_credit_limit",
    "total_rev_hi_lim",
]

## Feature Transformation Strategy

Feature transformation is defined after reviewing feature type, cardinality, and numeric distribution shape in the modeling population.

Two columns are excluded from the modeling feature space:

- `loan_status` is removed because it reflects repayment outcome and would introduce direct target leakage.
- `row_id` is excluded from the feature set but retained in the dataset to preserve row-level identity across pipeline stages and enable explicit alignment during validation.

---

Categorical transformation is limited to:

- `purpose`  
- `home_ownership`  

These are the only retained borrower-facing categorical features. Both are low-cardinality application-time attributes and can be one-hot encoded without materially increasing dimensionality.

The transformation design is intentionally branch-aware rather than uniform. Three model-input branches are constructed from the same post-drop base:

1. **Shared branch**  
   Selected monetary features are log-transformed, retained categoricals are one-hot encoded, and remaining missing values are imputed. This branch is used for Logistic Regression, Random Forest (shared), and CatBoost (shared) to enable direct comparison across model classes.

2. **Tree no-log branch**  
   The same categorical encoding and imputation steps are applied, but monetary features are left on their original scale. This branch isolates the impact of log transformation on tree-based models.

3. **CatBoost native branch**  
   Categorical features remain in native form, missing values are preserved, and monetary features are left untransformed. This branch evaluates CatBoost under a representation closer to its intended usage.

---

The following binary indicators are retained without transformation:

- `has_mths_since_last_delinq`  
- `has_mths_since_last_major_derog`  
- `has_mths_since_last_record`  
- `has_mths_since_recent_bc_dlq`  
- `has_mths_since_recent_revol_delinq`  

These variables already encode the presence or absence of specific credit events. As structurally binary features, additional transformation would not improve representation.

---

Log transformation is applied selectively to the following numeric features in the **shared branch only**:

- `annual_inc`  
- `loan_amnt`  
- `revol_bal`  
- `tot_coll_amt`  
- `total_bal_ex_mort`  
- `avg_cur_bal`  
- `bc_open_to_buy`  
- `tot_cur_bal`  
- `tot_hi_cred_lim`  
- `total_bc_limit`  
- `total_il_high_credit_limit`  
- `total_rev_hi_lim`  

These variables represent monetary exposure and exhibit substantial right skew. In their raw form, a small number of extreme observations dominate scale without proportionally increasing signal. Log transformation compresses the upper tail while preserving rank order, improving learnability for linear models and stabilizing feature influence.

---

The remaining numeric features are left untransformed for structural reasons:

1. **Count variables**  
   Delinquencies, inquiries, account totals, and public records represent discrete event frequency. Log transformation would reduce interpretability without clear modeling benefit.

2. **Ratio and bounded variables**  
   Variables such as `dti`, `revol_util`, `bc_util`, `percent_bc_gt_75`, and `pct_tl_nvr_dlq` already operate on constrained scales. Their primary structure is not driven by extreme magnitude, so log transformation does not address a meaningful limitation.

3. **Time-since variables**  
   Several variables use sentinel-coded values to represent missing historical events. Their skewness reflects data construction as much as borrower behavior, so log transformation would distort this encoding rather than improve representation.

4. **Lower-range numeric variables**  
   Terms, counts, and basic credit history measures remain directly interpretable and usable across all model classes in their original scale.

---

The resulting feature engineering design prioritizes controlled comparison over automation. Leakage and identifiers are removed from the feature set, while `row_id` is retained for alignment. From this common base, three branches are constructed to isolate the effect of transformation choices on model behavior.

---

In [19]:
#--------------------------------------------------------------------------------
# Create post-drop base dataframes for downstream feature-engineering branches
#--------------------------------------------------------------------------------

df_post_drop_train = fu.drop_columns_with_logging(
    df=df_feature_engineering_train,
    columns_to_drop=columns_to_drop,
    dataset_name="train",
    log=PROJECT_LOG_FILE,
)

df_post_drop_test = fu.drop_columns_with_logging(
    df=df_feature_engineering_test,
    columns_to_drop=columns_to_drop,
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

{
    "stage": "post_drop_base_created",
    "dropped_column_count": len(columns_to_drop),
    "dropped_columns": columns_to_drop,
    "post_drop_train_shape": df_post_drop_train.shape,
    "post_drop_test_shape": df_post_drop_test.shape,
}

{'stage': 'post_drop_base_created',
 'dropped_column_count': 1,
 'dropped_columns': ['loan_status'],
 'post_drop_train_shape': (282945, 66),
 'post_drop_test_shape': (60012, 66)}

In [20]:
#--------------------------------------------------------------------------------
# Create feature-engineering branches from the common post-drop base
#--------------------------------------------------------------------------------

df_shared_model_train = df_post_drop_train.copy()
df_shared_model_test = df_post_drop_test.copy()

df_tree_no_log_train = df_post_drop_train.copy()
df_tree_no_log_test = df_post_drop_test.copy()

df_catboost_native_train = df_post_drop_train.copy()
df_catboost_native_test = df_post_drop_test.copy()

{
    "stage": "feature_engineering_branches_created",
    "shared_model_train_shape": df_shared_model_train.shape,
    "shared_model_test_shape": df_shared_model_test.shape,
    "tree_no_log_train_shape": df_tree_no_log_train.shape,
    "tree_no_log_test_shape": df_tree_no_log_test.shape,
    "catboost_native_train_shape": df_catboost_native_train.shape,
    "catboost_native_test_shape": df_catboost_native_test.shape,
}

{'stage': 'feature_engineering_branches_created',
 'shared_model_train_shape': (282945, 66),
 'shared_model_test_shape': (60012, 66),
 'tree_no_log_train_shape': (282945, 66),
 'tree_no_log_test_shape': (60012, 66),
 'catboost_native_train_shape': (282945, 66),
 'catboost_native_test_shape': (60012, 66)}

In [21]:
# --------------------------------------------------------
# Apply numeric log transformation to shared branch only
# --------------------------------------------------------

df_shared_model_train = fe.apply_numeric_log_transformation(
    df=df_shared_model_train,
    feature_names=log_transform_features,
    dataset_name="train",
    log=PROJECT_LOG_FILE,
)

df_shared_model_test = fe.apply_numeric_log_transformation(
    df=df_shared_model_test,
    feature_names=log_transform_features,
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

{
    "stage": "shared_model_numeric_log_transformation_applied",
    "shared_model_train_shape": df_shared_model_train.shape,
    "shared_model_test_shape": df_shared_model_test.shape,
    "log_transform_feature_count": len(log_transform_features),
    "log_transform_features": log_transform_features,
}

{'stage': 'shared_model_numeric_log_transformation_applied',
 'shared_model_train_shape': (282945, 66),
 'shared_model_test_shape': (60012, 66),
 'log_transform_feature_count': 12,
 'log_transform_features': ['annual_inc',
  'loan_amnt',
  'revol_bal',
  'tot_coll_amt',
  'total_bal_ex_mort',
  'avg_cur_bal',
  'bc_open_to_buy',
  'tot_cur_bal',
  'tot_hi_cred_lim',
  'total_bc_limit',
  'total_il_high_credit_limit',
  'total_rev_hi_lim']}

In [22]:
# --------------------------------------------------------
# Build category mapping from common post-drop train data
# --------------------------------------------------------

categorical_features_for_encoding = [
    feature_name
    for feature_name in categorical_features
    if feature_name in df_post_drop_train.columns
]

categorical_feature_mapping = fe.build_category_mapping(
    df_train=df_post_drop_train,
    feature_names=categorical_features_for_encoding,
    log=PROJECT_LOG_FILE,
)

{
    "stage": "category_mapping_built",
    "categorical_feature_count": len(categorical_features_for_encoding),
    "categorical_features": categorical_features_for_encoding,
    "category_sizes": {
        feature_name: len(category_mapping)
        for feature_name, category_mapping in categorical_feature_mapping.items()
    },
}

{'stage': 'category_mapping_built',
 'categorical_feature_count': 2,
 'categorical_features': ['purpose', 'home_ownership'],
 'category_sizes': {'purpose': 14, 'home_ownership': 4}}

In [23]:
# --------------------------------------------------------
# Apply one-hot encoding to shared and tree no-log branches
# --------------------------------------------------------

df_shared_model_train = fe.apply_one_hot_encoding(
    df=df_shared_model_train,
    feature_names=categorical_features_for_encoding,
    dataset_name="train",
    category_mapping=categorical_feature_mapping,
    log=PROJECT_LOG_FILE,
)

df_shared_model_test = fe.apply_one_hot_encoding(
    df=df_shared_model_test,
    feature_names=categorical_features_for_encoding,
    dataset_name="test",
    category_mapping=categorical_feature_mapping,
    log=PROJECT_LOG_FILE,
)

df_tree_no_log_train = fe.apply_one_hot_encoding(
    df=df_tree_no_log_train,
    feature_names=categorical_features_for_encoding,
    dataset_name="train",
    category_mapping=categorical_feature_mapping,
    log=PROJECT_LOG_FILE,
)

df_tree_no_log_test = fe.apply_one_hot_encoding(
    df=df_tree_no_log_test,
    feature_names=categorical_features_for_encoding,
    dataset_name="test",
    category_mapping=categorical_feature_mapping,
    log=PROJECT_LOG_FILE,
)

{
    "stage": "one_hot_encoding_applied",
    "shared_model_train_shape": df_shared_model_train.shape,
    "shared_model_test_shape": df_shared_model_test.shape,
    "tree_no_log_train_shape": df_tree_no_log_train.shape,
    "tree_no_log_test_shape": df_tree_no_log_test.shape,
    "encoded_feature_count": len(categorical_features_for_encoding),
    "encoded_features": categorical_features_for_encoding,
    "catboost_native_categoricals_retained": True,
}

{'stage': 'one_hot_encoding_applied',
 'shared_model_train_shape': (282945, 82),
 'shared_model_test_shape': (60012, 82),
 'tree_no_log_train_shape': (282945, 82),
 'tree_no_log_test_shape': (60012, 82),
 'encoded_feature_count': 2,
 'encoded_features': ['purpose', 'home_ownership'],
 'catboost_native_categoricals_retained': True}

In [24]:
# --------------------------------------------------------
# Apply median imputation to branches that require imputed inputs
# --------------------------------------------------------

shared_imputation_features = [
    column_name
    for column_name in df_shared_model_train.columns
    if df_shared_model_train[column_name].isna().any()
]

tree_no_log_imputation_features = [
    column_name
    for column_name in df_tree_no_log_train.columns
    if df_tree_no_log_train[column_name].isna().any()
]

df_shared_model_train, df_shared_model_test = fe.apply_median_imputation(
    df_train=df_shared_model_train,
    df_test=df_shared_model_test,
    feature_names=shared_imputation_features,
    log=PROJECT_LOG_FILE,
)

df_tree_no_log_train, df_tree_no_log_test = fe.apply_median_imputation(
    df_train=df_tree_no_log_train,
    df_test=df_tree_no_log_test,
    feature_names=tree_no_log_imputation_features,
    log=PROJECT_LOG_FILE,
)

log(f"[median_imputation_applied][shared_features] {shared_imputation_features}")
log(f"[median_imputation_applied][tree_no_log_features] {tree_no_log_imputation_features}")

{
    "stage": "median_imputation_applied",
    "shared_imputed_feature_count": len(shared_imputation_features),
    "tree_no_log_imputed_feature_count": len(tree_no_log_imputation_features),
    "shared_model_train_shape": df_shared_model_train.shape,
    "shared_model_test_shape": df_shared_model_test.shape,
    "tree_no_log_train_shape": df_tree_no_log_train.shape,
    "tree_no_log_test_shape": df_tree_no_log_test.shape,
    "catboost_native_train_shape": df_catboost_native_train.shape,
    "catboost_native_test_shape": df_catboost_native_test.shape,
    "catboost_native_imputation_applied": False,
}

{'stage': 'median_imputation_applied',
 'shared_imputed_feature_count': 48,
 'tree_no_log_imputed_feature_count': 48,
 'shared_model_train_shape': (282945, 82),
 'shared_model_test_shape': (60012, 82),
 'tree_no_log_train_shape': (282945, 82),
 'tree_no_log_test_shape': (60012, 82),
 'catboost_native_train_shape': (282945, 66),
 'catboost_native_test_shape': (60012, 66),
 'catboost_native_imputation_applied': False}

In [25]:
# --------------------------------------------------------
# Validate engineered datasets
# --------------------------------------------------------

shared_model_train_columns = set(df_shared_model_train.columns)
shared_model_test_columns = set(df_shared_model_test.columns)

tree_no_log_train_columns = set(df_tree_no_log_train.columns)
tree_no_log_test_columns = set(df_tree_no_log_test.columns)

catboost_native_train_columns = set(df_catboost_native_train.columns)
catboost_native_test_columns = set(df_catboost_native_test.columns)

engineered_dataset_checkpoint = {
    "stage": "feature_engineering_complete",
    "shared_model": {
        "train_shape": df_shared_model_train.shape,
        "test_shape": df_shared_model_test.shape,
        "train_column_count": len(df_shared_model_train.columns),
        "test_column_count": len(df_shared_model_test.columns),
        "train_only_columns": sorted(shared_model_train_columns - shared_model_test_columns),
        "test_only_columns": sorted(shared_model_test_columns - shared_model_train_columns),
    },
    "tree_no_log": {
        "train_shape": df_tree_no_log_train.shape,
        "test_shape": df_tree_no_log_test.shape,
        "train_column_count": len(df_tree_no_log_train.columns),
        "test_column_count": len(df_tree_no_log_test.columns),
        "train_only_columns": sorted(tree_no_log_train_columns - tree_no_log_test_columns),
        "test_only_columns": sorted(tree_no_log_test_columns - tree_no_log_train_columns),
    },
    "catboost_native": {
        "train_shape": df_catboost_native_train.shape,
        "test_shape": df_catboost_native_test.shape,
        "train_column_count": len(df_catboost_native_train.columns),
        "test_column_count": len(df_catboost_native_test.columns),
        "train_only_columns": sorted(catboost_native_train_columns - catboost_native_test_columns),
        "test_only_columns": sorted(catboost_native_test_columns - catboost_native_train_columns),
    },
}

log(f"[feature_engineering_complete] {engineered_dataset_checkpoint}")

engineered_dataset_checkpoint

{'stage': 'feature_engineering_complete',
 'shared_model': {'train_shape': (282945, 82),
  'test_shape': (60012, 82),
  'train_column_count': 82,
  'test_column_count': 82,
  'train_only_columns': [],
  'test_only_columns': []},
 'tree_no_log': {'train_shape': (282945, 82),
  'test_shape': (60012, 82),
  'train_column_count': 82,
  'test_column_count': 82,
  'train_only_columns': [],
  'test_only_columns': []},
 'catboost_native': {'train_shape': (282945, 66),
  'test_shape': (60012, 66),
  'train_column_count': 66,
  'test_column_count': 66,
  'train_only_columns': [],
  'test_only_columns': []}}

In [26]:
# --------------------------------------------------------
# Final model columns artifacts
# --------------------------------------------------------

shared_model_columns_df = pd.DataFrame(
    {"feature_name": df_shared_model_train.columns.tolist()}
)
tree_no_log_columns_df = pd.DataFrame(
    {"feature_name": df_tree_no_log_train.columns.tolist()}
)
catboost_native_columns_df = pd.DataFrame(
    {"feature_name": df_catboost_native_train.columns.tolist()}
)

shared_model_columns_file = modeling_tables_dir / "shared_model_columns.csv"
tree_no_log_columns_file = modeling_tables_dir / "tree_no_log_columns.csv"
catboost_native_columns_file = modeling_tables_dir / "catboost_native_columns.csv"

shared_model_columns_df.to_csv(shared_model_columns_file, index=False)
tree_no_log_columns_df.to_csv(tree_no_log_columns_file, index=False)
catboost_native_columns_df.to_csv(catboost_native_columns_file, index=False)

log(
    "[feature_engineering][final_model_columns] "
    f"shared_model_file={shared_model_columns_file} "
    f"tree_no_log_file={tree_no_log_columns_file} "
    f"catboost_native_file={catboost_native_columns_file}"
)

{
    "stage": "final_model_columns_created",
    "shared_model_rows": int(shared_model_columns_df.shape[0]),
    "tree_no_log_rows": int(tree_no_log_columns_df.shape[0]),
    "catboost_native_rows": int(catboost_native_columns_df.shape[0]),
    "shared_model_output_path": str(shared_model_columns_file),
    "tree_no_log_output_path": str(tree_no_log_columns_file),
    "catboost_native_output_path": str(catboost_native_columns_file),
}

{'stage': 'final_model_columns_created',
 'shared_model_rows': 82,
 'tree_no_log_rows': 82,
 'catboost_native_rows': 66,
 'shared_model_output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\shared_model_columns.csv',
 'tree_no_log_output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\tree_no_log_columns.csv',
 'catboost_native_output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\catboost_native_columns.csv'}

In [27]:
# --------------------------------------------------------
# Validate final model branches
# --------------------------------------------------------

final_branch_checkpoint = {
    "shared_model_train_missing_values": int(df_shared_model_train.isna().sum().sum()),
    "shared_model_test_missing_values": int(df_shared_model_test.isna().sum().sum()),
    "tree_no_log_train_missing_values": int(df_tree_no_log_train.isna().sum().sum()),
    "tree_no_log_test_missing_values": int(df_tree_no_log_test.isna().sum().sum()),
    "catboost_native_train_missing_values": int(df_catboost_native_train.isna().sum().sum()),
    "catboost_native_test_missing_values": int(df_catboost_native_test.isna().sum().sum()),
}

log(f"[final_model_branch_validation] {final_branch_checkpoint}")

final_branch_checkpoint

{'shared_model_train_missing_values': 0,
 'shared_model_test_missing_values': 0,
 'tree_no_log_train_missing_values': 0,
 'tree_no_log_test_missing_values': 0,
 'catboost_native_train_missing_values': 2183204,
 'catboost_native_test_missing_values': 14336}

In [28]:
# --------------------------------------------------------
# Save final model datasets
# --------------------------------------------------------

df_shared_model_train.to_parquet(shared_model_train_data_file)
df_shared_model_test.to_parquet(shared_model_test_data_file)

df_tree_no_log_train.to_parquet(tree_no_log_train_data_file)
df_tree_no_log_test.to_parquet(tree_no_log_test_data_file)

df_catboost_native_train.to_parquet(catboost_native_train_data_file)
df_catboost_native_test.to_parquet(catboost_native_test_data_file)

log(f"[model_datasets_saved] shared_model_train_path={shared_model_train_data_file}")
log(f"[model_datasets_saved] shared_model_test_path={shared_model_test_data_file}")
log(f"[model_datasets_saved] tree_no_log_train_path={tree_no_log_train_data_file}")
log(f"[model_datasets_saved] tree_no_log_test_path={tree_no_log_test_data_file}")
log(f"[model_datasets_saved] catboost_native_train_path={catboost_native_train_data_file}")
log(f"[model_datasets_saved] catboost_native_test_path={catboost_native_test_data_file}")

{
    "stage": "model_datasets_saved",
    "shared_model_train_rows": int(df_shared_model_train.shape[0]),
    "shared_model_test_rows": int(df_shared_model_test.shape[0]),
    "shared_model_train_columns": int(df_shared_model_train.shape[1]),
    "shared_model_test_columns": int(df_shared_model_test.shape[1]),
    "tree_no_log_train_rows": int(df_tree_no_log_train.shape[0]),
    "tree_no_log_test_rows": int(df_tree_no_log_test.shape[0]),
    "tree_no_log_train_columns": int(df_tree_no_log_train.shape[1]),
    "tree_no_log_test_columns": int(df_tree_no_log_test.shape[1]),
    "catboost_native_train_rows": int(df_catboost_native_train.shape[0]),
    "catboost_native_test_rows": int(df_catboost_native_test.shape[0]),
    "catboost_native_train_columns": int(df_catboost_native_train.shape[1]),
    "catboost_native_test_columns": int(df_catboost_native_test.shape[1]),
}

{'stage': 'model_datasets_saved',
 'shared_model_train_rows': 282945,
 'shared_model_test_rows': 60012,
 'shared_model_train_columns': 82,
 'shared_model_test_columns': 82,
 'tree_no_log_train_rows': 282945,
 'tree_no_log_test_rows': 60012,
 'tree_no_log_train_columns': 82,
 'tree_no_log_test_columns': 82,
 'catboost_native_train_rows': 282945,
 'catboost_native_test_rows': 60012,
 'catboost_native_train_columns': 66,
 'catboost_native_test_columns': 66}

## Feature Engineering Conclusion

Feature engineering produced three distinct model-input branches from a common post-drop base.

The **shared branch** applies selective numeric log transformation, one-hot encoding, and median imputation. This branch provides the common comparison space used for Logistic Regression, Random Forest, and the shared CatBoost run.

The **tree no-log branch** applies the same one-hot encoding and median imputation steps, but leaves selected monetary features on their original scale. This branch isolates whether log transformation materially changes tree-model performance under otherwise comparable preprocessing.

The **CatBoost native branch** preserves retained categorical variables in native form, keeps missing values in place, and leaves selected monetary features untransformed. This branch tests CatBoost under a more native tree-based representation.

With these branches in place, the next stage is model training and comparative evaluation across both model class and preprocessing design.

---

## 3. Modeling

With the modeling population defined and feature engineering complete, the next step is to estimate predictive models on the engineered training data and evaluate how well they generalize to the testing period.

The modeling design now combines model-class comparison with preprocessing-branch comparison. Three branches are used:

- a **shared branch** with selective log transformation, one-hot encoding, and imputation
- a **tree no-log branch** with one-hot encoding and imputation but no numeric log transformation
- a **CatBoost native branch** with native categorical handling, native missing-value handling, and no numeric log transformation

Within this design, Logistic Regression is trained on the shared branch as a linear baseline, Random Forest is compared across the shared and tree no-log branches, and CatBoost is compared across the shared branch, the tree no-log branch, and the native branch.

This structure answers three related questions at once: how well default risk can be predicted from application-time information, how much predictive value is gained by moving from a linear baseline to more flexible model classes, and whether tree-based models benefit materially from preprocessing choices such as numeric log transformation and native categorical or missing-value handling.

---

In [29]:
# --------------------------------------------------------
# Split model inputs and target
# --------------------------------------------------------

target_column = "target_default"
row_identifier_column = "row_id"

X_shared_model_train = df_shared_model_train.drop(columns=[target_column, row_identifier_column]).copy()
y_shared_model_train = df_shared_model_train[target_column].copy()

X_shared_model_test = df_shared_model_test.drop(columns=[target_column, row_identifier_column]).copy()
y_shared_model_test = df_shared_model_test[target_column].copy()

X_tree_no_log_train = df_tree_no_log_train.drop(columns=[target_column, row_identifier_column]).copy()
y_tree_no_log_train = df_tree_no_log_train[target_column].copy()

X_tree_no_log_test = df_tree_no_log_test.drop(columns=[target_column, row_identifier_column]).copy()
y_tree_no_log_test = df_tree_no_log_test[target_column].copy()

X_catboost_native_train = df_catboost_native_train.drop(columns=[target_column, row_identifier_column]).copy()
y_catboost_native_train = df_catboost_native_train[target_column].copy()

X_catboost_native_test = df_catboost_native_test.drop(columns=[target_column, row_identifier_column]).copy()
y_catboost_native_test = df_catboost_native_test[target_column].copy()

{
    "stage": "model_inputs_split",
    "target_column": target_column,
    "row_identifier_column": row_identifier_column,
    "shared_model_X_train_shape": X_shared_model_train.shape,
    "shared_model_X_test_shape": X_shared_model_test.shape,
    "shared_model_y_train_shape": y_shared_model_train.shape,
    "shared_model_y_test_shape": y_shared_model_test.shape,
    "tree_no_log_X_train_shape": X_tree_no_log_train.shape,
    "tree_no_log_X_test_shape": X_tree_no_log_test.shape,
    "tree_no_log_y_train_shape": y_tree_no_log_train.shape,
    "tree_no_log_y_test_shape": y_tree_no_log_test.shape,
    "catboost_native_X_train_shape": X_catboost_native_train.shape,
    "catboost_native_X_test_shape": X_catboost_native_test.shape,
    "catboost_native_y_train_shape": y_catboost_native_train.shape,
    "catboost_native_y_test_shape": y_catboost_native_test.shape,
}

{'stage': 'model_inputs_split',
 'target_column': 'target_default',
 'row_identifier_column': 'row_id',
 'shared_model_X_train_shape': (282945, 80),
 'shared_model_X_test_shape': (60012, 80),
 'shared_model_y_train_shape': (282945,),
 'shared_model_y_test_shape': (60012,),
 'tree_no_log_X_train_shape': (282945, 80),
 'tree_no_log_X_test_shape': (60012, 80),
 'tree_no_log_y_train_shape': (282945,),
 'tree_no_log_y_test_shape': (60012,),
 'catboost_native_X_train_shape': (282945, 64),
 'catboost_native_X_test_shape': (60012, 64),
 'catboost_native_y_train_shape': (282945,),
 'catboost_native_y_test_shape': (60012,)}

In [30]:
# --------------------------------------------------------
# Identify CatBoost native categorical features
# --------------------------------------------------------

catboost_native_categorical_features = [
    column_name
    for column_name in X_catboost_native_train.columns
    if str(X_catboost_native_train[column_name].dtype) in {"category", "object"}
]

{
    "stage": "catboost_native_categorical_features_identified",
    "categorical_feature_count": len(catboost_native_categorical_features),
    "categorical_features": catboost_native_categorical_features,
}

{'stage': 'catboost_native_categorical_features_identified',
 'categorical_feature_count': 2,
 'categorical_features': ['home_ownership', 'purpose']}

In [31]:
# --------------------------------------------------------
# Train logistic regression model
# --------------------------------------------------------

logistic_regression_model, logistic_regression_train_metadata = tm.train_logistic_regression(
    X_train=X_shared_model_train,
    y_train=y_shared_model_train,
    log=PROJECT_LOG_FILE,
    random_state=42,
    max_iter=1000,
    class_weight=None,
    solver="lbfgs",
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {
                "warning_message": logistic_regression_train_metadata["warning_messages"]
            }
        )
    )

{
    "stage": "logistic_regression_trained",
    "training_branch": "shared_model",
    "model_class": type(logistic_regression_model).__name__,
    "train_rows": int(X_shared_model_train.shape[0]),
    "train_columns": int(X_shared_model_train.shape[1]),
    "warning_count": logistic_regression_train_metadata["warning_count"],
    "convergence_warning_count": logistic_regression_train_metadata["convergence_warning_count"],
}

,warning_message
0,ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):\nSTOP: TOTAL NO. OF ITERATIONS REACHED LIMIT\n\nIncrease the number of iterations to improve the convergence (max_iter=1000).\nYou might also want to scale the data as shown in:\n https://scikit-learn.org/stable/modules/preprocessing.html\nPlease also refer to the documentation for alternative solver options:\n https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression


{'stage': 'logistic_regression_trained',
 'training_branch': 'shared_model',
 'model_class': 'LogisticRegression',
 'train_rows': 282945,
 'train_columns': 80,
 'warning_count': 1,
 'convergence_warning_count': 1}

In [32]:
# --------------------------------------------------------
# Scale inputs for Logistic Regression
# --------------------------------------------------------

X_shared_model_train_logistic, X_shared_model_test_logistic, logistic_scaler = tm.scale_logistic_regression_inputs(
    X_train=X_shared_model_train,
    X_test=X_shared_model_test,
    log=PROJECT_LOG_FILE,
)

{
    "stage": "logistic_inputs_scaled",
    "training_branch": "shared_model",
    "X_shared_model_train_logistic_shape": X_shared_model_train_logistic.shape,
    "X_shared_model_test_logistic_shape": X_shared_model_test_logistic.shape,
}

{'stage': 'logistic_inputs_scaled',
 'training_branch': 'shared_model',
 'X_shared_model_train_logistic_shape': (282945, 80),
 'X_shared_model_test_logistic_shape': (60012, 80)}

In [33]:
# --------------------------------------------------------
# Train logistic regression model after scaling inputs
# --------------------------------------------------------

logistic_regression_model, logistic_regression_train_metadata = tm.train_logistic_regression(
    X_train=X_shared_model_train_logistic,
    y_train=y_shared_model_train,
    log=PROJECT_LOG_FILE,
    random_state=42,
    max_iter=2000,
    class_weight=None,
    solver="lbfgs",
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {
                "warning_message": logistic_regression_train_metadata["warning_messages"]
            }
        )
    )

{
    "stage": "logistic_regression_trained",
    "training_branch": "shared_model",
    "input_representation": "scaled_shared_model",
    "model_class": type(logistic_regression_model).__name__,
    "train_rows": int(X_shared_model_train_logistic.shape[0]),
    "train_columns": int(X_shared_model_train_logistic.shape[1]),
    "warning_count": logistic_regression_train_metadata["warning_count"],
    "convergence_warning_count": logistic_regression_train_metadata["convergence_warning_count"],
}

,warning_message


{'stage': 'logistic_regression_trained',
 'training_branch': 'shared_model',
 'input_representation': 'scaled_shared_model',
 'model_class': 'LogisticRegression',
 'train_rows': 282945,
 'train_columns': 80,
 'warning_count': 0,
 'convergence_warning_count': 0}

Logistic Regression was trained on a scaled version of the shared imputed feature matrix. This scaling step was applied only to the linear baseline because it supports stable numerical optimization and does not alter the underlying feature set used for model comparison.

In [34]:
# ---------------------------------------------------------------------------------------------
# Train Random Forest model (shared model branch with log transformation and imputation)
# ---------------------------------------------------------------------------------------------

shared_random_forest_model, random_forest_train_metadata = tm.train_random_forest(
    X_train=X_shared_model_train,
    y_train=y_shared_model_train,
    log=PROJECT_LOG_FILE,
    random_state=42,
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight=None,
    n_jobs=-1,
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {"warning_message": random_forest_train_metadata["warning_messages"]}
        )
    )
    
{
    "stage": "random_forest_trained",
    "model_class": type(shared_random_forest_model).__name__,
    "train_rows": int(X_shared_model_train.shape[0]),
    "train_columns": int(X_shared_model_train.shape[1]),
    "warning_count": random_forest_train_metadata["warning_count"],
    "n_estimators": random_forest_train_metadata["n_estimators"],
    "max_depth": random_forest_train_metadata["max_depth"],
    "max_features": random_forest_train_metadata["max_features"],
}

,warning_message


{'stage': 'random_forest_trained',
 'model_class': 'RandomForestClassifier',
 'train_rows': 282945,
 'train_columns': 80,
 'warning_count': 0,
 'n_estimators': 300,
 'max_depth': None,
 'max_features': 'sqrt'}

In [35]:
# ---------------------------------------------------------------------------------------------
# Train Random Forest model (No-log, but with imputation)
# ---------------------------------------------------------------------------------------------

tree_no_log_random_forest_model, random_forest_train_metadata = tm.train_random_forest(
    X_train=X_tree_no_log_train,
    y_train=y_tree_no_log_train,
    log=PROJECT_LOG_FILE,
    random_state=42,
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight=None,
    n_jobs=-1,
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {"warning_message": random_forest_train_metadata["warning_messages"]}
        )
    )
    
{
    "stage": "random_forest_trained",
    "model_class": type(tree_no_log_random_forest_model).__name__,
    "train_rows": int(X_tree_no_log_train.shape[0]),
    "train_columns": int(X_tree_no_log_train.shape[1]),
    "warning_count": random_forest_train_metadata["warning_count"],
    "n_estimators": random_forest_train_metadata["n_estimators"],
    "max_depth": random_forest_train_metadata["max_depth"],
    "max_features": random_forest_train_metadata["max_features"],
}

,warning_message


{'stage': 'random_forest_trained',
 'model_class': 'RandomForestClassifier',
 'train_rows': 282945,
 'train_columns': 80,
 'warning_count': 0,
 'n_estimators': 300,
 'max_depth': None,
 'max_features': 'sqrt'}

In [36]:
# -------------------------------------------------------------------------------------------------------------------
# Train CatBoost model (shared branch with one-hot encoding, median imputation, and numeric log transformation)
# -------------------------------------------------------------------------------------------------------------------

catboost_shared_model, catboost_shared_train_metadata = tm.train_catboost(
    X_train=X_shared_model_train,
    y_train=y_shared_model_train,
    log=PROJECT_LOG_FILE,
    random_state=42,
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=False,
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {"warning_message": catboost_shared_train_metadata["warning_messages"]}
        )
    )

{
    "stage": "catboost_shared_trained",
    "model_class": type(catboost_shared_model).__name__,
    "train_rows": int(X_shared_model_train.shape[0]),
    "train_columns": int(X_shared_model_train.shape[1]),
    "warning_count": catboost_shared_train_metadata["warning_count"],
    "iterations": catboost_shared_train_metadata["model_params"]["iterations"],
    "learning_rate": catboost_shared_train_metadata["model_params"]["learning_rate"],
    "depth": catboost_shared_train_metadata["model_params"]["depth"],
}

,warning_message


{'stage': 'catboost_shared_trained',
 'model_class': 'CatBoostClassifier',
 'train_rows': 282945,
 'train_columns': 80,
 'warning_count': 0,
 'iterations': 500,
 'learning_rate': 0.05,
 'depth': 6}

In [37]:
# ------------------------------------------------------------------------------------------------------
# Train CatBoost model (tree no-log branch with one-hot encoding and median imputation)
# ------------------------------------------------------------------------------------------------------

catboost_tree_no_log_model, catboost_tree_no_log_train_metadata = tm.train_catboost(
    X_train=X_tree_no_log_train,
    y_train=y_tree_no_log_train,
    log=PROJECT_LOG_FILE,
    random_state=42,
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=False,
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {"warning_message": catboost_tree_no_log_train_metadata["warning_messages"]}
        )
    )

{
    "stage": "catboost_tree_no_log_trained",
    "model_class": type(catboost_tree_no_log_model).__name__,
    "train_rows": int(X_tree_no_log_train.shape[0]),
    "train_columns": int(X_tree_no_log_train.shape[1]),
    "warning_count": catboost_tree_no_log_train_metadata["warning_count"],
    "iterations": catboost_tree_no_log_train_metadata["model_params"]["iterations"],
    "learning_rate": catboost_tree_no_log_train_metadata["model_params"]["learning_rate"],
    "depth": catboost_tree_no_log_train_metadata["model_params"]["depth"],
}

,warning_message


{'stage': 'catboost_tree_no_log_trained',
 'model_class': 'CatBoostClassifier',
 'train_rows': 282945,
 'train_columns': 80,
 'warning_count': 0,
 'iterations': 500,
 'learning_rate': 0.05,
 'depth': 6}

In [38]:
# --------------------------------------------------------
# Train CatBoost model with native categorical handling
# --------------------------------------------------------

catboost_native_model, catboost_native_train_metadata = tm.train_catboost(
    X_train=X_catboost_native_train,
    y_train=y_catboost_native_train,
    categorical_feature_names=catboost_native_categorical_features,
    log=PROJECT_LOG_FILE,
    random_state=42,
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=False,
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {"warning_message": catboost_native_train_metadata["warning_messages"]}
        )
    )

{
    "stage": "catboost_native_trained",
    "model_class": type(catboost_native_model).__name__,
    "train_rows": int(X_catboost_native_train.shape[0]),
    "train_columns": int(X_catboost_native_train.shape[1]),
    "warning_count": catboost_native_train_metadata["warning_count"],
    "iterations": catboost_native_train_metadata["model_params"]["iterations"],
    "learning_rate": catboost_native_train_metadata["model_params"]["learning_rate"],
    "depth": catboost_native_train_metadata["model_params"]["depth"],
}

,warning_message


{'stage': 'catboost_native_trained',
 'model_class': 'CatBoostClassifier',
 'train_rows': 282945,
 'train_columns': 64,
 'warning_count': 0,
 'iterations': 500,
 'learning_rate': 0.05,
 'depth': 6}

## Modeling Conclusion

The modeling stage successfully constructs and trains multiple model classes under the constraint that only application-time information is used. Logistic Regression, Random Forest, and CatBoost models are all implemented using the same feature space, ensuring that differences in performance reflect modeling approach rather than data leakage or inconsistent inputs.

Separate preprocessing branches are used where necessary, but all models operate on the same underlying information set. This allows for a controlled comparison in the evaluation phase.

At this stage, no conclusions are drawn about which model performs best. The purpose of the modeling step is to produce valid, comparable models that can be evaluated in terms of both predictive performance and decision impact.

The next step is therefore to evaluate these models on the test set and analyze their error structure and economic implications.

---

## 4. Evaluation

Model training establishes candidate estimators, but does not by itself answer how well they generalize beyond the training window. Evaluation is therefore conducted on the 2015 testing period in order to assess out-of-sample performance under the temporal split defined for the project.

The comparison includes Logistic Regression on the shared branch, Random Forest on both the shared and tree no-log branches, CatBoost on both the shared and tree no-log branches, and an additional CatBoost native run with native categorical and missing-value handling. This makes it possible to compare model classes under a common input structure while also testing two preprocessing questions directly: whether numeric log transformation materially affects tree-model performance, and whether CatBoost gains materially from a more native representation.

Evaluation focuses on three questions. First, how well do the models rank higher-risk borrowers above lower-risk borrowers? Second, how do their classification outcomes behave on the testing period? Third, how informative and stable are the predicted default probabilities? Together, these results determine which model and preprocessing branch provides the strongest basis for subsequent tuning and validation against LendingClub’s grading system.

---


In [39]:
# --------------------------------------------------------
# Statistical Evaluation Logistic Regression Model
# --------------------------------------------------------

logistic_regression_test_predictions_df = em.generate_model_predictions(
    model=logistic_regression_model,
    X_data=X_shared_model_test_logistic,
    dataset_name="test",
    model_name="logistic_regression",
    log=PROJECT_LOG_FILE,
    threshold=0.5,
)

logistic_regression_test_metrics_df = em.build_classification_metrics_table(
    y_true=y_shared_model_test,
    prediction_dataframe=logistic_regression_test_predictions_df,
    model_name="logistic_regression",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Statistical Evaluation Random Forest Shared Model
# --------------------------------------------------------

shared_random_forest_test_predictions_df = em.generate_model_predictions(
    model=shared_random_forest_model,
    X_data=X_shared_model_test,
    dataset_name="test",
    model_name="shared_random_forest",
    log=PROJECT_LOG_FILE,
    threshold=0.5,
)

shared_random_forest_test_metrics_df = em.build_classification_metrics_table(
    y_true=y_shared_model_test,
    prediction_dataframe=shared_random_forest_test_predictions_df,
    model_name="shared_random_forest",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Statistical Evaluation Random Forest Tree-No-Log Model
# --------------------------------------------------------

tree_no_log_random_forest_test_predictions_df = em.generate_model_predictions(
    model=tree_no_log_random_forest_model,
    X_data=X_tree_no_log_test,
    dataset_name="test",
    model_name="tree_no_log_random_forest",
    log=PROJECT_LOG_FILE,
    threshold=0.5,
)

tree_no_log_random_forest_test_metrics_df = em.build_classification_metrics_table(
    y_true=y_tree_no_log_test,
    prediction_dataframe=tree_no_log_random_forest_test_predictions_df,
    model_name="tree_no_log_random_forest",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Statistical Evaluation CatBoost Shared Model
# --------------------------------------------------------

catboost_shared_test_predictions_df = em.generate_model_predictions(
    model=catboost_shared_model,
    X_data=X_shared_model_test,
    dataset_name="test",
    model_name="catboost_shared",
    log=PROJECT_LOG_FILE,
    threshold=0.5,
)

catboost_shared_test_metrics_df = em.build_classification_metrics_table(
    y_true=y_shared_model_test,
    prediction_dataframe=catboost_shared_test_predictions_df,
    model_name="catboost_shared",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Statistical Evaluation CatBoost Tree-No-Log Model
# --------------------------------------------------------

catboost_tree_no_log_test_predictions_df = em.generate_model_predictions(
    model=catboost_tree_no_log_model,
    X_data=X_tree_no_log_test,
    dataset_name="test",
    model_name="catboost_tree_no_log",
    log=PROJECT_LOG_FILE,
    threshold=0.5,
)

catboost_tree_no_log_test_metrics_df = em.build_classification_metrics_table(
    y_true=y_tree_no_log_test,
    prediction_dataframe=catboost_tree_no_log_test_predictions_df,
    model_name="catboost_tree_no_log",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Statistical Evaluation CatBoost Native Model
# --------------------------------------------------------

catboost_native_test_predictions_df = em.generate_model_predictions(
    model=catboost_native_model,
    X_data=X_catboost_native_test,
    dataset_name="test",
    model_name="catboost_native",
    log=PROJECT_LOG_FILE,
    threshold=0.5,
)

catboost_native_test_metrics_df = em.build_classification_metrics_table(
    y_true=y_catboost_native_test,
    prediction_dataframe=catboost_native_test_predictions_df,
    model_name="catboost_native",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Combine model metrics into single table
# --------------------------------------------------------

model_comparison_metrics_df = em.combine_metrics_tables(
    metrics_tables=[
        logistic_regression_test_metrics_df,
        shared_random_forest_test_metrics_df,
        tree_no_log_random_forest_test_metrics_df,
        catboost_shared_test_metrics_df,
        catboost_tree_no_log_test_metrics_df,
        catboost_native_test_metrics_df,
    ],
    log=PROJECT_LOG_FILE,
)

display(model_comparison_metrics_df)

model_comparison_metrics_file = modeling_tables_dir / "model_comparison_metrics.csv"
model_comparison_metrics_df.to_csv(model_comparison_metrics_file, index=False)

log(f"[evaluation][model_comparison_metrics] table_file={model_comparison_metrics_file}")

{
    "stage": "model_comparison_metrics_created",
    "rows": int(model_comparison_metrics_df.shape[0]),
    "model_count": int(model_comparison_metrics_df["model_name"].nunique()),
    "output_path": str(model_comparison_metrics_file),
}

,model_name,dataset_name,roc_auc,accuracy,precision,recall,f1,brier_score,true_negative,false_positive,false_negative,true_positive
0,logistic_regression,test,0.705828,0.811304,0.469626,0.107987,0.175597,0.140517,47482,1362,9962,1206
1,shared_random_forest,test,0.709380,0.814121,0.506341,0.046472,0.085131,0.141013,48338,506,10649,519
2,tree_no_log_random_forest,test,0.709370,0.814071,0.504883,0.046293,0.084810,0.141010,48337,507,10651,517
3,catboost_shared,test,0.722055,0.812254,0.482747,0.124015,0.197336,0.138251,47360,1484,9783,1385
4,catboost_tree_no_log,test,0.722113,0.812787,0.488175,0.123836,0.197557,0.138233,47394,1450,9785,1383
5,catboost_native,test,0.723023,0.811421,0.474668,0.125000,0.197888,0.138188,47299,1545,9772,1396


{'stage': 'model_comparison_metrics_created',
 'rows': 6,
 'model_count': 6,
 'output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\model_comparison_metrics.csv'}

### Why discrimination metrics are not sufficient

The primary evaluation metric used so far is ROC AUC. This metric measures how well the model ranks borrowers from low risk to high risk across all possible thresholds. A higher ROC AUC indicates that, on average, borrowers who default receive higher predicted probabilities than those who do not.

However, this does not directly translate into better lending decisions. Lending is not concerned with ranking alone, but with specific actions: which loans are approved and which are rejected. Those actions depend on a chosen threshold, and different thresholds produce different combinations of errors.

In particular, two models with very similar ROC AUC values can produce meaningfully different error structures at a given threshold. One model may allow more defaults to pass through (false negatives), while another may reject more good loans (false positives). These differences are not captured by ROC AUC but are central to the economic outcome of the model.

For this reason, model evaluation must go beyond discrimination metrics and examine the structure and cost of prediction errors.


## Evaluation Progress and Next Step

The first stage of evaluation compared model performance on the 2015 testing period using standard classification metrics. This included ROC AUC, accuracy, precision, recall, F1, Brier score, and confusion matrix counts. Together, these results showed how well each model ranked borrower risk, how conservative or aggressive its default classifications were at the default threshold, and how informative its predicted probabilities were.

At this stage, CatBoost remained the strongest overall model family, but the branch comparison now matters as much as the model comparison. The results distinguish between a shared branch designed for direct comparability, a tree no-log branch designed to isolate the effect of numeric log transformation on tree models, and a native CatBoost branch that preserves categorical and missing-value structure more directly. This makes it possible to assess not only which model performs best, but also whether preprocessing choices materially change that conclusion.

These results establish which models perform best in statistical terms, but they do not yet show the full decision meaning of those errors. The confusion matrix counts treat every loan equally, even though a false negative on a small loan and a false negative on a much larger loan do not carry the same practical consequence. In the same way, a model that appears weaker in event counts could still be preferable if the loans it misclassifies are economically less important.

The next step therefore extends evaluation from counts to exposure. For each model, the predicted outcomes will be linked back to raw loan amounts in the testing data so that true positives, true negatives, false positives, and false negatives can also be assessed in monetary terms. This makes it possible to examine not only which model performs best statistically, but also which model performs best when errors are viewed through the size of the loans involved.

---

## Feature Space Assessment

The current results also do not support opening a broader feature-engineering cycle.

Performance gains across model classes are incremental rather than structural. Moving from Logistic Regression to Random Forest and then to CatBoost improves predictive performance, but not by an amount that suggests a large volume of unused signal remains in the existing feature set. This matters because different model classes extract different kinds of structure. Logistic Regression can only capture relatively simple linear relationships. Random Forest can capture non-linear thresholds and interactions. CatBoost is more flexible again and is usually better at exploiting subtle interactions and irregular variable behavior. If major predictive structure were still sitting in the current variables but simply had not been represented properly, the shift from a linear model to stronger tree-based models would normally produce a clearer performance jump. That is not what happens here. Performance improves, but it improves gradually. The models are not revealing a hidden reserve of signal waiting to be unlocked by more elaborate feature construction. This pattern is consistent with established results in statistical learning, where improvements in predictive performance tend to exhibit diminishing returns as model complexity increases (Ng, 2000).

The branch comparisons support the same conclusion. Removing numeric log transformation does not materially change tree-model performance, and CatBoost does not gain meaningfully from a more native representation. That matters because these branches directly test whether current feature handling is suppressing useful information. If the main limitation were a poor representation of the existing variables, these preprocessing changes would be expected to move the results more clearly. Instead, they leave the model ranking largely unchanged and produce only marginal differences in scores. Taken together, this suggests that the feature representation is not the main constraint.

The error structure points in the same direction. False negatives remain present, but not in a pattern that indicates an obvious missing interaction or overlooked relationship that could be recovered through additional feature construction. In practice, that means the models are still missing some defaults, but they are not doing so in a way that clearly identifies a specific blind spot in the current feature space. There is no obvious sign that a small number of ratio features, interaction terms, or alternative transforms would resolve a concentrated weakness. Instead, the remaining errors look more like the result of limited information than of an unmodeled pattern sitting in plain sight.

Under the strict application-time boundary used in this project, that result is plausible. The model only sees borrower and loan information available at the moment of application. It does not see later repayment behavior, servicing developments, changing borrower circumstances, or other post-origination signals that may drive eventual default. That means some uncertainty is structural. It reflects the limits of what can be known at origination, not a failure to engineer the current variables aggressively enough. In statistical learning terms, this corresponds to irreducible error in the data-generating process, where remaining prediction error cannot be eliminated through additional modeling or feature construction (Hastie et al., 2009).

For that reason, further feature work would add complexity without materially improving the decision boundary. The feature space is therefore treated as sufficient. The next step is to lock the strongest model configuration, apply controlled hyperparameter tuning, test whether loan-size weighting improves decision usefulness, and then prepare the final version of the model produced in this notebook before moving the broader threshold and portfolio-level decision analysis into the separate validation notebook.

---

### Interpreting prediction errors in a lending context

The confusion matrix distinguishes between four outcomes, but in a lending setting these outcomes have direct economic meaning.

- **True Negative (TN):** A non-defaulting loan that is correctly approved. This represents successful lending activity.
- **True Positive (TP):** A defaulting loan that is correctly rejected. This represents avoided loss.
- **False Positive (FP):** A non-defaulting loan that is rejected. This represents missed opportunity.
- **False Negative (FN):** A defaulting loan that is approved. This represents realized loss.

Because loan amounts differ across observations, the economic impact of these errors is not uniform. A single false negative on a large loan can outweigh many small errors. For this reason, the analysis aggregates total loan amounts within each outcome category to approximate the financial impact of model decisions.


In [40]:
# --------------------------------------------------------
# Prepare raw loan amounts for monetary evaluation
# --------------------------------------------------------

loan_amounts_test = df_model_population_test["loan_amnt"].copy()

{
    "stage": "loan_amounts_prepared",
    "rows": int(loan_amounts_test.shape[0]),
    "missing_values": int(loan_amounts_test.isna().sum()),
}

{'stage': 'loan_amounts_prepared', 'rows': 60012, 'missing_values': 0}

In [41]:
# --------------------------------------------------------
# Monetary Evaluation Logistic Regression Model
# --------------------------------------------------------

logistic_regression_outcome_df = em.build_prediction_outcome_table(
    y_true=y_shared_model_test,
    prediction_dataframe=logistic_regression_test_predictions_df,
    loan_amounts=loan_amounts_test,
    model_name="logistic_regression",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

logistic_regression_loan_outcome_summary_df = em.summarize_outcomes_by_loan_amount(
    outcome_dataframe=logistic_regression_outcome_df,
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Monetary Evaluation Random Forest Shared Model
# --------------------------------------------------------

shared_random_forest_outcome_df = em.build_prediction_outcome_table(
    y_true=y_shared_model_test,
    prediction_dataframe=shared_random_forest_test_predictions_df,
    loan_amounts=loan_amounts_test,
    model_name="shared_random_forest",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

shared_random_forest_loan_outcome_summary_df = em.summarize_outcomes_by_loan_amount(
    outcome_dataframe=shared_random_forest_outcome_df,
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Monetary Evaluation Random Forest Tree-No-Log Model
# --------------------------------------------------------

tree_no_log_random_forest_outcome_df = em.build_prediction_outcome_table(
    y_true=y_tree_no_log_test,
    prediction_dataframe=tree_no_log_random_forest_test_predictions_df,
    loan_amounts=loan_amounts_test,
    model_name="tree_no_log_random_forest",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

tree_no_log_random_forest_loan_outcome_summary_df = em.summarize_outcomes_by_loan_amount(
    outcome_dataframe=tree_no_log_random_forest_outcome_df,
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Monetary Evaluation CatBoost Shared Model
# --------------------------------------------------------

catboost_shared_outcome_df = em.build_prediction_outcome_table(
    y_true=y_shared_model_test,
    prediction_dataframe=catboost_shared_test_predictions_df,
    loan_amounts=loan_amounts_test,
    model_name="catboost_shared",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

catboost_shared_loan_outcome_summary_df = em.summarize_outcomes_by_loan_amount(
    outcome_dataframe=catboost_shared_outcome_df,
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Monetary Evaluation CatBoost Tree-No-Log Model
# --------------------------------------------------------

catboost_tree_no_log_outcome_df = em.build_prediction_outcome_table(
    y_true=y_tree_no_log_test,
    prediction_dataframe=catboost_tree_no_log_test_predictions_df,
    loan_amounts=loan_amounts_test,
    model_name="catboost_tree_no_log",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

catboost_tree_no_log_loan_outcome_summary_df = em.summarize_outcomes_by_loan_amount(
    outcome_dataframe=catboost_tree_no_log_outcome_df,
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Monetary Evaluation CatBoost Native Model
# --------------------------------------------------------

catboost_native_outcome_df = em.build_prediction_outcome_table(
    y_true=y_catboost_native_test,
    prediction_dataframe=catboost_native_test_predictions_df,
    loan_amounts=loan_amounts_test,
    model_name="catboost_native",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

catboost_native_loan_outcome_summary_df = em.summarize_outcomes_by_loan_amount(
    outcome_dataframe=catboost_native_outcome_df,
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Combine loan outcome summaries into single table
# --------------------------------------------------------

loan_outcome_comparison_df = pd.concat(
    [
        logistic_regression_loan_outcome_summary_df,
        shared_random_forest_loan_outcome_summary_df,
        tree_no_log_random_forest_loan_outcome_summary_df,
        catboost_shared_loan_outcome_summary_df,
        catboost_tree_no_log_loan_outcome_summary_df,
        catboost_native_loan_outcome_summary_df,
    ],
    axis=0,
    ignore_index=True,
)

loan_outcome_comparison_display_df = loan_outcome_comparison_df.copy()

money_columns = [
    "total_loan_amnt",
    "average_loan_amnt",
    "median_loan_amnt",
]

for column_name in money_columns:
    loan_outcome_comparison_display_df[column_name] = (
        loan_outcome_comparison_display_df[column_name]
        .map(lambda value: f"${value:,.0f}")
    )

display(loan_outcome_comparison_display_df)

loan_outcome_comparison_file = modeling_tables_dir / "loan_outcome_comparison.csv"
loan_outcome_comparison_df.to_csv(loan_outcome_comparison_file, index=False)

log(f"[evaluation][loan_outcome_comparison] table_file={loan_outcome_comparison_file}")

{
    "stage": "loan_outcome_comparison_created",
    "rows": int(loan_outcome_comparison_df.shape[0]),
    "model_count": int(loan_outcome_comparison_df["model_name"].nunique()),
    "output_path": str(loan_outcome_comparison_file),
}

,model_name,dataset_name,outcome_type,row_count,total_loan_amnt,average_loan_amnt,median_loan_amnt
0,logistic_regression,test,false_negative,9962,"$151,714,900","$15,229","$13,600"
1,logistic_regression,test,false_positive,1362,"$24,344,425","$17,874","$16,000"
2,logistic_regression,test,true_negative,47482,"$681,641,800","$14,356","$12,000"
3,logistic_regression,test,true_positive,1206,"$20,861,225","$17,298","$15,650"
4,shared_random_forest,test,false_negative,10649,"$163,442,400","$15,348","$14,000"
5,shared_random_forest,test,false_positive,506,"$9,103,150","$17,990","$16,512"
6,shared_random_forest,test,true_negative,48338,"$696,883,075","$14,417","$12,000"
7,shared_random_forest,test,true_positive,519,"$9,133,725","$17,599","$16,150"
8,tree_no_log_random_forest,test,false_negative,10651,"$163,484,150","$15,349","$14,000"
9,tree_no_log_random_forest,test,false_positive,507,"$9,157,975","$18,063","$16,625"


{'stage': 'loan_outcome_comparison_created',
 'rows': 24,
 'model_count': 6,
 'output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\loan_outcome_comparison.csv'}

## Monetary Evaluation

Monetary evaluation extends the confusion-matrix comparison by linking predicted outcomes in the testing period back to raw loan amounts. This shows not only how often each model makes different types of classification errors, but also how much loan exposure is attached to those errors.

This step matters because the statistical comparison alone cannot tell whether a model that looks weaker in event counts is still preferable once loan size is taken into account. A false negative on a small loan and a false negative on a much larger loan count the same in the confusion matrix, but they do not carry the same practical consequence. The same logic applies to false positives: some models may reject more good loans in count terms while still protecting more or less economically important exposure.

The monetary comparison therefore evaluates the same model-and-branch combinations used in the statistical comparison — Logistic Regression on the shared branch, Random Forest on the shared and tree no-log branches, CatBoost on the shared and tree no-log branches, and CatBoost on the native branch — but now in terms of the raw principal amounts attached to true positives, true negatives, false positives, and false negatives.

---

### Interpreting the model trade-off

The comparison between CatBoost variants illustrates an important trade-off.

The native CatBoost configuration achieves the lowest false negative exposure, meaning it allows slightly fewer defaulting loans to pass through. However, the magnitude of this improvement is small relative to the increase in false positive exposure, meaning a larger volume of good loans is rejected.

The shared CatBoost model, while marginally worse in terms of false negative exposure, avoids rejecting as many good loans. This results in a more balanced error profile. The preferred model is therefore not the one that minimizes a single metric, but the one that provides the best balance between avoiding losses and preserving lending opportunity.

---

## Evaluation Figures

In [42]:
# --------------------------------------------------------
# Generate ROC curve comparison figure
# --------------------------------------------------------

roc_figure_path = rf.plot_model_roc_comparison_figure(
    model_curves=[
        {
            "model_name": "logistic_regression",
            "y_true": y_shared_model_test,
            "y_score": logistic_regression_test_predictions_df["predicted_probability"],
        },
        {
            "model_name": "catboost_shared",
            "y_true": y_shared_model_test,
            "y_score": catboost_shared_test_predictions_df["predicted_probability"],
        },
        {
            "model_name": "catboost_native",
            "y_true": y_catboost_native_test,
            "y_score": catboost_native_test_predictions_df["predicted_probability"],
        },
    ],
    output_path=modeling_figures_dir / "roc_curve_comparison.png",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Generate calibration curve figure
# --------------------------------------------------------

calibration_figure_path = rf.plot_model_calibration_figure(
    y_true=y_shared_model_test,
    y_score=catboost_shared_test_predictions_df["predicted_probability"],
    model_name="catboost_shared",
    output_path=modeling_figures_dir / "catboost_shared_calibration.png",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Generate confusion matrix figure
# --------------------------------------------------------

confusion_matrix_figure_path = rf.plot_confusion_matrix_figure(
    y_true=y_shared_model_test,
    y_pred=catboost_shared_test_predictions_df["predicted_label"],
    model_name="catboost_shared",
    output_path=modeling_figures_dir / "catboost_shared_confusion_matrix.png",
    labels=[0, 1],
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Generate monetary outcome comparison figure
# --------------------------------------------------------

monetary_figure_path = rf.plot_monetary_outcome_comparison_figure(
    outcome_summary_df=loan_outcome_comparison_df,
    selected_models=["catboost_shared", "catboost_native"],
    output_path=modeling_figures_dir / "catboost_monetary_outcome_comparison.png",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Figures checkpoint output
# --------------------------------------------------------

{
    "stage": "evaluation_figures_created",
    "roc_figure_path": str(roc_figure_path),
    "calibration_figure_path": str(calibration_figure_path),
    "confusion_matrix_figure_path": str(confusion_matrix_figure_path),
    "monetary_figure_path": str(monetary_figure_path),
}

{'stage': 'evaluation_figures_created',
 'roc_figure_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\figures\\roc_curve_comparison.png',
 'calibration_figure_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\figures\\catboost_shared_calibration.png',
 'confusion_matrix_figure_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\figures\\catboost_shared_confusion_matrix.png',
 'monetary_figure_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\figures\\catboost_monetary_outcome_comparison.png'}

## 5. Model Selection and Optimization

The statistical comparison identified `catboost_native` as the strongest model on ROC AUC, Brier score, recall, and false-negative count. The monetary comparison, however, slightly changes the decision. While `catboost_native` produces the lowest false-negative exposure, its advantage over `catboost_shared` is negligible in dollar terms, while its false-positive exposure is materially higher. In other words, the native branch catches only a very small additional amount of bad lending, but does so at the cost of rejecting meaningfully more good lending.

For that reason, the shared CatBoost branch is selected as the candidate model for optimization. It remains one of the strongest statistical models, preserves a cleaner and more comparable preprocessing design, and provides the more favorable balance between avoided losses and unnecessarily rejected good loans.

The next step is therefore to optimize `catboost_shared` in two stages. First, hyperparameter tuning is used to improve the untuned candidate without changing the objective. Second, a weighted version of the tuned model is estimated so that larger loans receive more influence during training. This makes it possible to separate predictive optimization from objective redesign.

---

In [43]:
# --------------------------------------------------------
# Prepare raw training loan amounts for weighting
# --------------------------------------------------------

loan_amounts_train = df_model_population_train["loan_amnt"].loc[X_shared_model_train.index].copy()

{
    "stage": "loan_amounts_train_prepared",
    "rows": int(loan_amounts_train.shape[0]),
    "missing_values": int(loan_amounts_train.isna().sum()),
}

{'stage': 'loan_amounts_train_prepared', 'rows': 282945, 'missing_values': 0}

### CatBoost Shared Hyperparameter Tuning

Hyperparameter tuning is applied only to the selected candidate model. The goal is not to reopen model selection, but to improve the chosen CatBoost shared branch under the same predictive objective. A lightweight random search is used on a validation split drawn from the training data, leaving the testing period untouched until final evaluation.

---

In [44]:
# --------------------------------------------------------
# Tune CatBoost shared hyperparameters
# --------------------------------------------------------

best_catboost_shared_params, catboost_shared_tuning_results_df, catboost_shared_tuning_train_metadata = tm.tune_catboost_hyperparameters(
    X_train=X_shared_model_train,
    y_train=y_shared_model_train,
    log=PROJECT_LOG_FILE,
    random_state=42,
    n_trials=25,
    validation_size=0.2,
)

display(catboost_shared_tuning_results_df.head(10))

catboost_shared_tuning_results_file = (
    modeling_tables_dir / "catboost_shared_tuning_results.csv"
)

catboost_shared_tuning_results_df.to_csv(
    catboost_shared_tuning_results_file,
    index=False,
)

log(
    f"[model_optimization][catboost_shared_tuning_results] "
    f"table_file={catboost_shared_tuning_results_file}"
)

{
    "stage": "catboost_shared_tuning_completed",
    "trials": int(catboost_shared_tuning_results_df.shape[0]),
    "best_roc_auc": float(catboost_shared_tuning_train_metadata["best_roc_auc"]),
    "output_path": str(catboost_shared_tuning_results_file),
}

,trial_index,roc_auc,warning_count,iterations,learning_rate,depth,l2_leaf_reg,border_count,random_strength,warning_messages
0,3,0.719405,0,622,0.077644,6,5.991263,54,0.595726,
1,2,0.719003,0,720,0.068204,6,9.340885,107,1.734142,
2,20,0.718664,0,769,0.110522,4,5.015406,48,0.952268,
3,16,0.718622,0,797,0.050412,8,1.524725,115,0.940391,
4,23,0.718575,0,433,0.121238,5,5.044254,101,0.908362,
5,7,0.718450,0,673,0.062357,8,5.226002,67,0.784207,
6,0,0.718334,0,344,0.077054,7,8.727381,40,0.641266,
7,10,0.718241,0,518,0.070372,8,3.594953,55,0.709629,
8,14,0.718230,0,339,0.102513,7,5.982215,40,0.955925,
9,11,0.718217,0,641,0.045988,8,8.082319,32,1.497276,


{'stage': 'catboost_shared_tuning_completed',
 'trials': 25,
 'best_roc_auc': 0.7194053267583342,
 'output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\catboost_shared_tuning_results.csv'}

In [45]:
# --------------------------------------------------------
# Train tuned CatBoost shared model
# --------------------------------------------------------

catboost_shared_tuned_model, catboost_shared_tuned_train_metadata = tm.train_catboost(
    X_train=X_shared_model_train,
    y_train=y_shared_model_train,
    log=PROJECT_LOG_FILE,
    extra_params=best_catboost_shared_params,
    verbose=False,
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {"warning_message": catboost_shared_tuned_train_metadata["warning_messages"]}
        )
    )

{
    "stage": "catboost_shared_tuned_trained",
    "model_class": type(catboost_shared_tuned_model).__name__,
    "train_rows": int(X_shared_model_train.shape[0]),
    "train_columns": int(X_shared_model_train.shape[1]),
    "warning_count": catboost_shared_tuned_train_metadata["warning_count"],
}

,warning_message


{'stage': 'catboost_shared_tuned_trained',
 'model_class': 'CatBoostClassifier',
 'train_rows': 282945,
 'train_columns': 80,
 'warning_count': 0}

In [46]:
# --------------------------------------------------------
# Statistical evaluation for tuned CatBoost shared model
# --------------------------------------------------------

catboost_shared_tuned_test_predictions_df = em.generate_model_predictions(
    model=catboost_shared_tuned_model,
    X_data=X_shared_model_test,
    dataset_name="test",
    model_name="catboost_shared_tuned",
    log=PROJECT_LOG_FILE,
    threshold=0.5,
)

catboost_shared_tuned_test_metrics_df = em.build_classification_metrics_table(
    y_true=y_shared_model_test,
    prediction_dataframe=catboost_shared_tuned_test_predictions_df,
    model_name="catboost_shared_tuned",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

display(catboost_shared_tuned_test_metrics_df)

,model_name,dataset_name,roc_auc,accuracy,precision,recall,f1,brier_score,true_negative,false_positive,false_negative,true_positive
0,catboost_shared_tuned,test,0.723228,0.811638,0.479141,0.139864,0.216523,0.138333,47146,1698,9606,1562


In [47]:
# --------------------------------------------------------
# Monetary evaluation for tuned CatBoost shared model
# --------------------------------------------------------

catboost_shared_tuned_outcome_df = em.build_prediction_outcome_table(
    y_true=y_shared_model_test,
    prediction_dataframe=catboost_shared_tuned_test_predictions_df,
    loan_amounts=loan_amounts_test,
    model_name="catboost_shared_tuned",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

catboost_shared_tuned_loan_outcome_summary_df = em.summarize_outcomes_by_loan_amount(
    outcome_dataframe=catboost_shared_tuned_outcome_df,
    log=PROJECT_LOG_FILE,
)

display(catboost_shared_tuned_loan_outcome_summary_df)

,model_name,dataset_name,outcome_type,row_count,total_loan_amnt,average_loan_amnt,median_loan_amnt
0,catboost_shared_tuned,test,false_negative,9606,143811225,14970.979076,13075.0
1,catboost_shared_tuned,test,false_positive,1698,32327150,19038.368669,18000.0
2,catboost_shared_tuned,test,true_negative,47146,673659075,14288.785369,12000.0
3,catboost_shared_tuned,test,true_positive,1562,28764900,18415.428937,17112.5


### Loan-Amount Weighting

The tuned CatBoost shared model is now re-estimated with sample weights based on raw loan amounts. This does not change the feature space or the tuned parameter values. It changes only the training objective by giving larger loans more influence. The purpose of this step is to test whether the candidate model can be shifted toward better protection against economically larger mistakes without first changing the lending threshold.

In [48]:
# --------------------------------------------------------
# Build loan-amount weights for tuned CatBoost shared model
# --------------------------------------------------------

loan_amount_train_weights = tm.build_loan_amount_weights(
    loan_amounts=loan_amounts_train,
    method="sqrt",
    log=PROJECT_LOG_FILE,
)

{
    "stage": "loan_amount_weights_created",
    "rows": int(loan_amount_train_weights.shape[0]),
}

{'stage': 'loan_amount_weights_created', 'rows': 282945}

In [49]:
# --------------------------------------------------------
# Train tuned and weighted CatBoost shared model
# --------------------------------------------------------

catboost_shared_tuned_weighted_model, catboost_shared_tuned_weighted_train_metadata = tm.train_catboost(
    X_train=X_shared_model_train,
    y_train=y_shared_model_train,
    log=PROJECT_LOG_FILE,
    sample_weight=loan_amount_train_weights,
    extra_params=best_catboost_shared_params,
    verbose=False,
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {
                "warning_message": (
                    catboost_shared_tuned_weighted_train_metadata["warning_messages"]
                )
            }
        )
    )

{
    "stage": "catboost_shared_tuned_weighted_trained",
    "model_class": type(catboost_shared_tuned_weighted_model).__name__,
    "train_rows": int(X_shared_model_train.shape[0]),
    "train_columns": int(X_shared_model_train.shape[1]),
    "warning_count": catboost_shared_tuned_weighted_train_metadata["warning_count"],
}

,warning_message


{'stage': 'catboost_shared_tuned_weighted_trained',
 'model_class': 'CatBoostClassifier',
 'train_rows': 282945,
 'train_columns': 80,
 'warning_count': 0}

In [50]:
# --------------------------------------------------------
# Statistical evaluation for tuned and weighted CatBoost shared model
# --------------------------------------------------------

catboost_shared_tuned_weighted_test_predictions_df = em.generate_model_predictions(
    model=catboost_shared_tuned_weighted_model,
    X_data=X_shared_model_test,
    dataset_name="test",
    model_name="catboost_shared_tuned_weighted",
    log=PROJECT_LOG_FILE,
    threshold=0.5,
)

catboost_shared_tuned_weighted_test_metrics_df = em.build_classification_metrics_table(
    y_true=y_shared_model_test,
    prediction_dataframe=catboost_shared_tuned_weighted_test_predictions_df,
    model_name="catboost_shared_tuned_weighted",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

display(catboost_shared_tuned_weighted_test_metrics_df)

,model_name,dataset_name,roc_auc,accuracy,precision,recall,f1,brier_score,true_negative,false_positive,false_negative,true_positive
0,catboost_shared_tuned_weighted,test,0.723485,0.811438,0.476553,0.13467,0.209997,0.138162,47192,1652,9664,1504


### Why loan-amount weighting did not improve the model

The weighted model was designed to give larger loans more influence during training. The intuition behind this approach is that mistakes on larger loans are more costly.

While this adjustment slightly improved overall discrimination metrics, it did not improve the economic outcome relative to the tuned unweighted model. In particular, it did not reduce false negative exposure sufficiently and led to a deterioration in the error balance.

This suggests that the model was already capturing the relevant structure of risk in the data. Increasing the influence of larger loans did not introduce new information, but instead shifted the optimization in a way that did not translate into better decisions.

As a result, the weighted model is not selected.

---


In [51]:
# --------------------------------------------------------
# Monetary evaluation for tuned and weighted CatBoost shared model
# --------------------------------------------------------

catboost_shared_tuned_weighted_outcome_df = em.build_prediction_outcome_table(
    y_true=y_shared_model_test,
    prediction_dataframe=catboost_shared_tuned_weighted_test_predictions_df,
    loan_amounts=loan_amounts_test,
    model_name="catboost_shared_tuned_weighted",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

catboost_shared_tuned_weighted_loan_outcome_summary_df = em.summarize_outcomes_by_loan_amount(
    outcome_dataframe=catboost_shared_tuned_weighted_outcome_df,
    log=PROJECT_LOG_FILE,
)

display(catboost_shared_tuned_weighted_loan_outcome_summary_df)

,model_name,dataset_name,outcome_type,row_count,total_loan_amnt,average_loan_amnt,median_loan_amnt
0,catboost_shared_tuned_weighted,test,false_negative,9664,144760325,14979.338266,13087.5
1,catboost_shared_tuned_weighted,test,false_positive,1652,31926200,19325.786925,18000.0
2,catboost_shared_tuned_weighted,test,true_negative,47192,674060025,14283.353640,12000.0
3,catboost_shared_tuned_weighted,test,true_positive,1504,27815800,18494.547872,17187.5


In [52]:
# --------------------------------------------------------
# Compare baseline, tuned, and tuned+weighted CatBoost shared models
# --------------------------------------------------------

catboost_shared_model_optimization_metrics_df = em.combine_metrics_tables(
    metrics_tables=[
        catboost_shared_test_metrics_df,
        catboost_shared_tuned_test_metrics_df,
        catboost_shared_tuned_weighted_test_metrics_df,
    ],
    log=PROJECT_LOG_FILE,
)

display(catboost_shared_model_optimization_metrics_df)

catboost_shared_model_optimization_metrics_file = (
    modeling_tables_dir / "catboost_shared_model_optimization_metrics.csv"
)

catboost_shared_model_optimization_metrics_df.to_csv(
    catboost_shared_model_optimization_metrics_file,
    index=False,
)

log(
    f"[model_optimization][catboost_shared_model_optimization_metrics] "
    f"table_file={catboost_shared_model_optimization_metrics_file}"
)

{
    "stage": "catboost_shared_model_optimization_metrics_created",
    "rows": int(catboost_shared_model_optimization_metrics_df.shape[0]),
    "output_path": str(catboost_shared_model_optimization_metrics_file),
}

,model_name,dataset_name,roc_auc,accuracy,precision,recall,f1,brier_score,true_negative,false_positive,false_negative,true_positive
0,catboost_shared,test,0.722055,0.812254,0.482747,0.124015,0.197336,0.138251,47360,1484,9783,1385
1,catboost_shared_tuned,test,0.723228,0.811638,0.479141,0.139864,0.216523,0.138333,47146,1698,9606,1562
2,catboost_shared_tuned_weighted,test,0.723485,0.811438,0.476553,0.134670,0.209997,0.138162,47192,1652,9664,1504


{'stage': 'catboost_shared_model_optimization_metrics_created',
 'rows': 3,
 'output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\catboost_shared_model_optimization_metrics.csv'}

In [53]:
# --------------------------------------------------------
# Compare baseline, tuned, and tuned+weighted CatBoost shared monetary outcomes
# --------------------------------------------------------

catboost_shared_model_optimization_loan_outcomes_df = pd.concat(
    [
        catboost_shared_loan_outcome_summary_df,
        catboost_shared_tuned_loan_outcome_summary_df,
        catboost_shared_tuned_weighted_loan_outcome_summary_df,
    ],
    axis=0,
    ignore_index=True,
)

catboost_shared_model_optimization_loan_outcomes_display_df = (
    catboost_shared_model_optimization_loan_outcomes_df.copy()
)

for column_name in ["total_loan_amnt", "average_loan_amnt", "median_loan_amnt"]:
    catboost_shared_model_optimization_loan_outcomes_display_df[column_name] = (
        catboost_shared_model_optimization_loan_outcomes_display_df[column_name]
        .map(lambda value: f"${value:,.0f}")
    )

display(catboost_shared_model_optimization_loan_outcomes_display_df)

catboost_shared_model_optimization_loan_outcomes_file = (
    modeling_tables_dir / "catboost_shared_model_optimization_loan_outcomes.csv"
)

catboost_shared_model_optimization_loan_outcomes_df.to_csv(
    catboost_shared_model_optimization_loan_outcomes_file,
    index=False,
)

log(
    f"[model_optimization][catboost_shared_model_optimization_loan_outcomes] "
    f"table_file={catboost_shared_model_optimization_loan_outcomes_file}"
)

{
    "stage": "catboost_shared_model_optimization_loan_outcomes_created",
    "rows": int(catboost_shared_model_optimization_loan_outcomes_df.shape[0]),
    "output_path": str(catboost_shared_model_optimization_loan_outcomes_file),
}

,model_name,dataset_name,outcome_type,row_count,total_loan_amnt,average_loan_amnt,median_loan_amnt
0,catboost_shared,test,false_negative,9783,"$147,006,450","$15,027","$13,200"
1,catboost_shared,test,false_positive,1484,"$28,164,600","$18,979","$17,612"
2,catboost_shared,test,true_negative,47360,"$677,821,625","$14,312","$12,000"
3,catboost_shared,test,true_positive,1385,"$25,569,675","$18,462","$17,000"
4,catboost_shared_tuned,test,false_negative,9606,"$143,811,225","$14,971","$13,075"
5,catboost_shared_tuned,test,false_positive,1698,"$32,327,150","$19,038","$18,000"
6,catboost_shared_tuned,test,true_negative,47146,"$673,659,075","$14,289","$12,000"
7,catboost_shared_tuned,test,true_positive,1562,"$28,764,900","$18,415","$17,112"
8,catboost_shared_tuned_weighted,test,false_negative,9664,"$144,760,325","$14,979","$13,088"
9,catboost_shared_tuned_weighted,test,false_positive,1652,"$31,926,200","$19,326","$18,000"


{'stage': 'catboost_shared_model_optimization_loan_outcomes_created',
 'rows': 12,
 'output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\catboost_shared_model_optimization_loan_outcomes.csv'}

## Model Optimization Conclusion

The CatBoost shared model was selected as the candidate model because it offered the best balance between statistical performance and monetary trade-offs. Hyperparameter tuning then tested whether this candidate could be improved without changing the underlying objective.

The tuning step produced a measurable improvement. The model’s ability to separate higher-risk borrowers from lower-risk borrowers increased, recall improved, and the number of false negatives declined relative to the untuned model. This indicates that the model was not yet fully optimized and that controlled parameter tuning can meaningfully improve performance without altering the feature space.

A weighted variant was then estimated in order to test whether giving larger loans more influence during training would shift the model toward a more economically favorable error profile. While this adjustment led to a slight improvement in overall risk separation, it did not improve the practical lending objective. In particular, it did not reduce false negative exposure sufficiently and, in some cases, worsened the balance between false negatives and false positives.

This result suggests that the model already captures the relevant risk structure present in the data, and that increasing the influence of larger loans does not introduce additional usable signal. Instead, it shifts the optimization in a way that does not translate into better portfolio-level outcomes.

The optimization stage therefore leads to a clear selection: the final model from this notebook is the tuned, unweighted CatBoost shared model. This model is saved as a documented artifact and passed to the validation stage, where predictions are translated into decision thresholds and lending-policy trade-offs rather than treated as a pure classification exercise.

---

In [54]:
# --------------------------------------------------------
# Register and Save final selected model artifact
# --------------------------------------------------------

# ---- Select tuned model metrics row ----
selected_model_metrics_row = catboost_shared_model_optimization_metrics_df.loc[
    (catboost_shared_model_optimization_metrics_df["model_name"] == "catboost_shared_tuned")
    & (catboost_shared_model_optimization_metrics_df["dataset_name"] == "test")
].iloc[0]

# ---- Select tuned model monetary outcome rows ----
selected_model_fn_exposure = catboost_shared_model_optimization_loan_outcomes_df.loc[
    (catboost_shared_model_optimization_loan_outcomes_df["model_name"] == "catboost_shared_tuned")
    & (catboost_shared_model_optimization_loan_outcomes_df["outcome_type"] == "false_negative"),
    "total_loan_amnt",
].iloc[0]

selected_model_fp_exposure = catboost_shared_model_optimization_loan_outcomes_df.loc[
    (catboost_shared_model_optimization_loan_outcomes_df["model_name"] == "catboost_shared_tuned")
    & (catboost_shared_model_optimization_loan_outcomes_df["outcome_type"] == "false_positive"),
    "total_loan_amnt",
].iloc[0]

# ---- Extract actual fitted parameters from train metadata ----
selected_model_params = catboost_shared_tuned_train_metadata["model_params"]

# ---- Build metadata ----
selected_model_metadata = {
    "model_name": "catboost_shared_tuned",
    "notebook": "03_modeling",
    "model_type": "CatBoostClassifier",
    "preprocessing_branch": "shared",
    "tuned": True,
    "weighted": False,
    "selected_for_validation": True,
    "selection_reason": (
        "Selected based on the strongest balance between statistical performance "
        "and monetary trade-offs. Hyperparameter tuning improved the shared CatBoost "
        "model meaningfully, while the weighted variant did not improve the economic "
        "objective relative to the tuned unweighted model."
    ),
    "test_roc_auc": float(selected_model_metrics_row["roc_auc"]),
    "test_accuracy": float(selected_model_metrics_row["accuracy"]),
    "test_precision": float(selected_model_metrics_row["precision"]),
    "test_recall": float(selected_model_metrics_row["recall"]),
    "test_f1": float(selected_model_metrics_row["f1"]),
    "test_brier_score": float(selected_model_metrics_row["brier_score"]),
    "true_negative": int(selected_model_metrics_row["true_negative"]),
    "false_positive": int(selected_model_metrics_row["false_positive"]),
    "false_negative": int(selected_model_metrics_row["false_negative"]),
    "true_positive": int(selected_model_metrics_row["true_positive"]),
    "false_negative_exposure": float(selected_model_fn_exposure),
    "false_positive_exposure": float(selected_model_fp_exposure),
    "train_rows": int(X_shared_model_train.shape[0]),
    "test_rows": int(X_shared_model_test.shape[0]),
    "feature_count": int(X_shared_model_train.shape[1]),
    "model_params": selected_model_params,
}

# ---- Save model artifact ----
selected_model_artifact_path = sm.save_model(
    model=catboost_shared_tuned_model,
    output_path=selected_model_output_path,
    metadata=selected_model_metadata,
    log=PROJECT_LOG_FILE,
)

{
    "stage": "final_model_artifact_registered",
    "model_path": str(selected_model_artifact_path),
    "metadata_path": str(selected_model_output_path.with_suffix(".metadata.json")),
}
 

{'stage': 'final_model_artifact_registered',
 'model_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\models\\catboost_shared_tuned.joblib',
 'metadata_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\models\\catboost_shared_tuned.metadata.json'}

In [55]:
# -----------------------------------------------------------------------
# Save datasets of the selected model for handoff to validation
# -----------------------------------------------------------------------

df_selected_model_train = df_shared_model_train.copy()
df_selected_model_test = df_shared_model_test.copy()

df_selected_model_train.to_parquet(selected_model_input_train_file)
df_selected_model_test.to_parquet(selected_model_input_test_file)

log(
    {
        "stage": "model_handoff_save",
        "train_rows": df_selected_model_train.shape[0],
        "test_rows": df_selected_model_test.shape[0],
        "train_cols": df_selected_model_train.shape[1],
        "test_cols": df_selected_model_test.shape[1],
        "train_path": str(selected_model_input_train_file),
        "test_path": str(selected_model_input_test_file),
    }
)

# Save metadata
metadata = {
    "timestamp_utc": datetime.utcnow().isoformat(),
    "selected_model": "catboost_shared_tuned",
    "model_family": "CatBoost",
    "preprocessing_branch": "shared_model",
    "target_column": target_column,
    "train_data_path": str(selected_model_input_train_file),
    "test_data_path": str(selected_model_input_test_file),
    "notes": "Final selected model input dataset used for validation",
}

with open(metadata_file, "w") as f:
    json.dump(metadata, f, indent=4)

log(
    {
        "stage": "model_handoff_metadata_saved",
        "metadata_path": str(metadata_file),
    }
)

{
    "stage": "model_handoff_complete",
    "train_path": str(selected_model_input_train_file),
    "test_path": str(selected_model_input_test_file),
    "metadata_path": str(metadata_file),
    "train_rows": df_selected_model_train.shape[0],
    "test_rows": df_selected_model_test.shape[0],
    "row_id_present": "row_id" in df_selected_model_train.columns,
    "target_present": target_column in df_selected_model_train.columns,
}

{'stage': 'model_handoff_complete',
 'train_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\data\\processed\\selected_model_input_train_data.parquet',
 'test_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\data\\processed\\selected_model_input_test_data.parquet',
 'metadata_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\data\\processed\\selected_model_metadata.json',
 'train_rows': 282945,
 'test_rows': 60012,
 'row_id_present': True,
 'target_present': True}

## Notebook Conclusion

This notebook evaluates how well default risk can be captured using only application-time borrower information and whether increased modeling complexity improves predictive and decision outcomes.

The results show that default risk is meaningfully predictable, but not to a level that suggests near-perfect separation. Model performance improves as flexibility increases, with CatBoost outperforming both Logistic Regression and Random Forest. However, these improvements are incremental rather than structural, indicating that the available feature set already captures most of the accessible signal.

Model selection cannot be based on statistical metrics alone. While the CatBoost native configuration achieves slightly stronger risk separation and marginally lower false negative exposure, it does so at the cost of rejecting a materially larger volume of good loans. The shared CatBoost model provides a more balanced trade-off between risk reduction and lending opportunity and is therefore selected as the candidate model.

This candidate was then optimized. Hyperparameter tuning produced a measurable improvement, increasing risk separation, improving recall, and reducing the number of false negatives relative to the untuned model. This confirms that the model was not yet fully optimized and that controlled tuning can improve performance without changing the underlying feature space.

A weighted variant was also evaluated in order to prioritize larger loans during training. While this approach slightly improved overall risk separation, it did not improve the economic error profile relative to the tuned unweighted model. In particular, it did not reduce false negative exposure sufficiently and, in some cases, worsened the balance between false negatives and false positives. As a result, the weighted model is not selected.

The modeling stage therefore concludes with a clear outcome: the final model is the tuned, unweighted CatBoost shared model. This model represents the best balance between predictive performance and decision impact under the constraint of application-time information and is saved as a documented artifact.

The next step is to move beyond model performance and evaluate how predictions translate into decisions. This is addressed in the validation stage, where thresholds, trade-offs, and portfolio-level outcomes are analyzed.